# مدل دو مرحله‌ای گوجه: ۱) تشخیص گوجه  ۲) رسیدگی (raw / half_ripe / ripe / rotten) — Google Colab

## اجرا
1. این نوت‌بوک را در Google Drive بگذار و با Google Colaboratory باز کن ▸ `Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save`
2. کلید Roboflow از قبل در بلوک تنظیمات گذاشته شده است (برای دانلود دیتاست‌ها).
3. `Runtime ▸ Run all` — دیتاست‌ها خودکار دانلود می‌شوند: **Tomato Defects** (۴ کلاس)، **tomato-sxsc7**، **tomato_count_project** (نوار نقاله) + داده‌ی کمکی papapeto، tomato-detection-sorting، tomato-classification
4. **قطع شد؟** فقط دوباره `Run all`: هیچ کاری دوبار انجام نمی‌شود (دانلود، holdout، آموزش و پیش‌بینی‌ها روی Drive ذخیره‌اند و آموزش از آخرین epoch ادامه می‌دهد).

| بلوک | کار |
|---|---|
| ⚙️ | تنظیمات (مدل سبک YOLO26n/s، epoch، اندازه‌ی تصویر) |
| ۱ | آماده‌سازی Colab |
| ۲ | دانلود داده (یک بار)، جدا کردن ثابت ≥۳۰ عکس نوار نقاله + holdout از sxsc7 (هرگز آموزش نمی‌بینند)، حذف کپی‌های test از آموزش، ساخت داده‌ی دو مرحله، بررسی نشتی |
| ۳ | وضعیت آموزش |
| ۴ | مرحله‌ی ۱: آموزش تشخیص گوجه با **پنل زنده** (نوار پیشرفت، loss، دقت هر epoch، نمودار، زمان باقی‌مانده، GPU) |
| ۴.ب | مرحله‌ی ۲: آموزش رسیدگی روی برش هر گوجه |
| ۵ | نتیجه‌ی ۴ کلاسه روی test |
| ۶ | **ارزیابی class-agnostic** روی سه مجموعه + جدول اندازه‌ی شیء + بررسی recall ≥ ۰٫۸۵ + نمونه‌ی خطاها |
| ۷ | تست روی عکس خودت |


In [ ]:
#@title ⚙️ تنظیمات
MODEL_SIZE  = "yolo26s.pt"   #@param ["yolo26n.pt", "yolo26s.pt"]
CLS_MODEL   = "yolo26s-cls.pt"   #@param ["yolo26n-cls.pt", "yolo26s-cls.pt"]
CLS_IMGSZ   = 160   #@param {type:"integer"}
CLS_EPOCHS  = 60   #@param {type:"integer"}
EPOCHS      = 100   #@param {type:"integer"}
IMGSZ       = 640   #@param {type:"integer"}
PATIENCE    = 40   #@param {type:"integer"}
BATCH       = 0   #@param {type:"integer"}
SAVE_EVERY  = 2   #@param {type:"integer"}
QUICK_TEST  = False   #@param {type:"boolean"}
DRIVE_DIR   = "/content/drive/MyDrive/tomato_phase5"   #@param {type:"string"}
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"   #@param {type:"string"}
# yolo26n: lightest/fastest | yolo26s: still light, better on small far tomatoes (recommended)
MODEL = "tomato"
RECIPE = "t2"
RUN_IMGSZ, RUN_EPOCHS = (320, 2) if QUICK_TEST else (int(IMGSZ), int(EPOCHS))
RUN_NAME = f"{MODEL_SIZE[:-3]}_{RUN_IMGSZ}_{RECIPE}"
print("مدل:", MODEL_SIZE, "| imgsz", RUN_IMGSZ, "| epochs", RUN_EPOCHS, "| پوشه:", RUN_NAME,
      "| آزمایش سریع" if QUICK_TEST else "| آموزش کامل")

## 1) آماده‌سازی: Drive، GPU، کتابخانه‌ها، کدها

In [ ]:
#@title 1) آماده‌سازی (کدها پنهان هستند)
import os, sys, subprocess, glob, json, shutil, zipfile, base64, time, re, signal, threading, collections, importlib
WORK = "/content/work"                        # local fast disk (wiped on disconnect; results live on Drive)
os.makedirs(WORK, exist_ok=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def run_live(cmd):
    """Run a short command and stream its output into the cell (progress bars collapse to their last state)."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    try:
        for raw in p.stdout:                  # a progress bar redraws with \r: keep only its final state
            parts = [s for s in raw.decode("utf-8", "replace").rstrip("\r\n").split("\r") if s.strip()]
            if parts:
                print(parts[-1], flush=True)
    except KeyboardInterrupt:
        p.kill()
        raise
    if p.wait() != 0:
        raise RuntimeError(f"دستور با خطا تمام شد (کد {p.returncode}) - چند خط بالاتر علت را نوشته است")

# Python packages: the exact ultralytics version this code was tested with
try:
    import ultralytics
    have = ultralytics.__version__
except ImportError:
    have = None
if have != "8.4.14":
    run_live([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.14"])
try:
    import roboflow, pycocotools
except ImportError:
    run_live([sys.executable, "-m", "pip", "install", "-q", "roboflow", "pycocotools"])
import ultralytics, torch
importlib.reload(ultralytics)
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| python", sys.version.split()[0])

if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name} ({g.total_memory / 1e9:.0f} GB) | CPU cores: {os.cpu_count()}")
else:
    print("⚠️ GPU پیدا نشد: آموزش خودکار روی CPU انجام می‌شود (خیلی کندتر). برای سرعت: "
          "Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save و دوباره Run all")
print(f"فضای آزاد دیسک Colab: {shutil.disk_usage(WORK).free / 1e9:.0f} GB")

# the scripts (verify / train / predict / live panel), written next to each other on the local disk
SCRIPTS = {"train_tomato.py": "...", "panel.py": "...", "predict.py": "IiIiRmluZCBQQ0IgZGVmZWN0cyBvbiB5b3VyIG93biBwaG90b3MgKGFuIGltYWdlIG9yIGEgZm9sZGVyKS4KCkxhcmdlIHBob3RvcyBhcmUgY3V0IGludG8gb3ZlcmxhcHBpbmcgdGlsZXMgKGFzIGluIHRyYWluaW5nKSwgZXZlcnkgdGlsZSBpcyBwcmVkaWN0ZWQsIGFuZCB0aGUKYm94ZXMgYXJlIG1lcmdlZCB3aXRoIE5NUywgc28gYSAzMDAwIHB4IHBob3RvIGtlZXBzIGl0cyBmdWxsIGRldGFpbC4gT3V0cHV0cyBwZXIgaW1hZ2U6CjxuYW1lPl9hbm5vdGF0ZWQuanBnIGFuZCA8bmFtZT4uanNvbiAoY2xhc3MsIGNvbmZpZGVuY2UsIGJveCBpbiBwaXhlbHMpOyBzdW1tYXJ5LmNzdiBmb3IgYWxsLgoKQ29uZmlkZW5jZTogYnkgZGVmYXVsdCB0aGUgcGVyLWNsYXNzIHRocmVzaG9sZHMgb2YgdGhyZXNob2xkcy5qc29uIG5leHQgdG8gdGhlIHdlaWdodHMgKHR1bmVkIG9uIHZhbCBieQp0cmFpbl9zb2xkZXIucHkpOyAtLWNvbmYgWCB1c2VzIG9uZSB0aHJlc2hvbGQgWCBmb3IgZXZlcnkgY2xhc3MgaW5zdGVhZC4KLS1sYWJlbHMgPGZvbGRlciBvZiBZT0xPIC50eHQ+OiBhbHNvIGRyYXdzIHRoZSB0cnVlIGJveGVzICh0aGluIGdyZWVuKSB0byBjb21wYXJlIHdpdGggdGhlIHByZWRpY3Rpb25zLgoKICBweXRob24gcHJlZGljdC5weSAtLXdlaWdodHMgYmVzdC5wdCAtLXNvdXJjZSBteV9ib2FyZC5qcGcgLS1vdXQgcmVzdWx0cwoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNodmlzaW9uCmZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KCkNPTCA9IFsoMCwgMCwgMjU1KSwgKDI1NSwgMTI4LCAwKSwgKDI1NSwgMCwgMjU1KSwgKDAsIDIxNSwgMjU1KSwgKDI1NSwgMjU1LCAwKSwgKDEyOCwgMCwgMjU1KSwgKDAsIDEyOCwgMjU1KSwKICAgICAgICgxODAsIDEwNSwgMjU1KSwgKDQwLCAxODAsIDEyMCldCkdUID0gKDYwLCAyMjAsIDYwKQoKCmRlZiB0aWxlcyh3LCBoLCB0LCBvdmVybGFwKToKICAgIHN0ZXAgPSBpbnQodCAqICgxIC0gb3ZlcmxhcCkpCiAgICB4cyA9IGxpc3QocmFuZ2UoMCwgbWF4KDEsIHcgLSB0KSwgc3RlcCkpICsgW21heCgwLCB3IC0gdCldCiAgICB5cyA9IGxpc3QocmFuZ2UoMCwgbWF4KDEsIGggLSB0KSwgc3RlcCkpICsgW21heCgwLCBoIC0gdCldCiAgICByZXR1cm4gc29ydGVkKHsoeCwgeSkgZm9yIHggaW4geHMgZm9yIHkgaW4geXN9KQoKCmRlZiBwcmVkaWN0X2ltYWdlKG1vZGVsLCBpbWcsIGEsIGNvbmZfbWluKToKICAgIGgsIHcgPSBpbWcuc2hhcGVbOjJdCiAgICBqb2JzID0gWygwLCAwLCBpbWcpXSBpZiBtYXgoaCwgdykgPD0gYS50aWxlX2Fib3ZlIGVsc2UgXAogICAgICAgIFsoeCwgeSwgaW1nW3k6eSArIGEudGlsZSwgeDp4ICsgYS50aWxlXSkgZm9yIHgsIHkgaW4gdGlsZXModywgaCwgYS50aWxlLCBhLm92ZXJsYXApXQogICAgYm94ZXMsIHNjb3JlcywgY2xhc3NlcyA9IFtdLCBbXSwgW10KICAgIGZvciBrIGluIHJhbmdlKDAsIGxlbihqb2JzKSwgYS5iYXRjaCk6CiAgICAgICAgY2h1bmsgPSBqb2JzW2s6ayArIGEuYmF0Y2hdCiAgICAgICAgcmVzID0gbW9kZWwucHJlZGljdChbY1syXSBmb3IgYyBpbiBjaHVua10sIGltZ3N6PWEuaW1nc3osIGNvbmY9Y29uZl9taW4sIGlvdT1hLmlvdSwgdmVyYm9zZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRldmljZT1hLmRldmljZSkKICAgICAgICBmb3IgKHgwLCB5MCwgXyksIHIgaW4gemlwKGNodW5rLCByZXMpOgogICAgICAgICAgICBiID0gci5ib3hlcwogICAgICAgICAgICBpZiBsZW4oYik6CiAgICAgICAgICAgICAgICBib3hlcy5hcHBlbmQoYi54eXh5LmNwdSgpICsgdG9yY2gudGVuc29yKFt4MCwgeTAsIHgwLCB5MF0pKQogICAgICAgICAgICAgICAgc2NvcmVzLmFwcGVuZChiLmNvbmYuY3B1KCkpCiAgICAgICAgICAgICAgICBjbGFzc2VzLmFwcGVuZChiLmNscy5jcHUoKSkKICAgIGlmIG5vdCBib3hlczoKICAgICAgICByZXR1cm4gW10KICAgIEIsIFMsIEMgPSB0b3JjaC5jYXQoYm94ZXMpLCB0b3JjaC5jYXQoc2NvcmVzKSwgdG9yY2guY2F0KGNsYXNzZXMpCiAgICBrZWVwID0gdG9yY2h2aXNpb24ub3BzLmJhdGNoZWRfbm1zKEIsIFMsIEMsIGEuaW91KSBpZiBsZW4oam9icykgPiAxIGVsc2UgdG9yY2guYXJhbmdlKGxlbihTKSkKICAgIHJldHVybiBbKGludChDW2ldKSwgZmxvYXQoU1tpXSksICpbZmxvYXQodikgZm9yIHYgaW4gQltpXV0pIGZvciBpIGluIGtlZXBdCgoKZGVmIHJlYWRfZ3QobGJsLCB3LCBoKToKICAgIG91dCA9IFtdCiAgICBpZiBsYmwgYW5kIGxibC5leGlzdHMoKToKICAgICAgICBmb3IgbG4gaW4gbGJsLnJlYWRfdGV4dChlbmNvZGluZz0idXRmOCIpLnNwbGl0KCJcbiIpOgogICAgICAgICAgICBwID0gbG4uc3BsaXQoKQogICAgICAgICAgICBpZiBsZW4ocCkgPT0gNToKICAgICAgICAgICAgICAgIGMsIHgsIHksIGJ3LCBiaCA9IGludChwWzBdKSwgKm1hcChmbG9hdCwgcFsxOl0pCiAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKChjLCAoeCAtIGJ3IC8gMikgKiB3LCAoeSAtIGJoIC8gMikgKiBoLCAoeCArIGJ3IC8gMikgKiB3LCAoeSArIGJoIC8gMikgKiBoKSkKICAgIHJldHVybiBvdXQKCgpkZWYgZHJhdyhpbWcsIGRldHMsIGd0LCBuYW1lcyk6CiAgICB2aXMgPSBpbWcuY29weSgpCiAgICB0aCA9IG1heCgyLCBtYXgoaW1nLnNoYXBlWzoyXSkgLy8gNTAwKQogICAgZm9yIGMsIHgwLCB5MCwgeDEsIHkxIGluIGd0OgogICAgICAgIGN2Mi5yZWN0YW5nbGUodmlzLCAoaW50KHgwKSwgaW50KHkwKSksIChpbnQoeDEpLCBpbnQoeTEpKSwgR1QsIG1heCgxLCB0aCAvLyAyKSwgY3YyLkxJTkVfQUEpCiAgICBmcyA9IDAuNDUgKiB0aAogICAgZm9yIGMsIHMsIHgwLCB5MCwgeDEsIHkxIGluIHNvcnRlZChkZXRzLCBrZXk9bGFtYmRhIGQ6IGRbMV0pOgogICAgICAgIGNvbCA9IENPTFtjICUgbGVuKENPTCldCiAgICAgICAgcDAsIHAxID0gKGludCh4MCkgLSB0aCwgaW50KHkwKSAtIHRoKSwgKGludCh4MSkgKyB0aCwgaW50KHkxKSArIHRoKQogICAgICAgIGN2Mi5yZWN0YW5nbGUodmlzLCBwMCwgcDEsIGNvbCwgdGgsIGN2Mi5MSU5FX0FBKQogICAgICAgIHR4dCA9IGYie25hbWVzW2NdfSB7czouMmZ9IgogICAgICAgICh0dywgdGh0KSwgXyA9IGN2Mi5nZXRUZXh0U2l6ZSh0eHQsIGN2Mi5GT05UX0hFUlNIRVlfU0lNUExFWCwgZnMsIG1heCgxLCB0aCAvLyAyKSkKICAgICAgICB0eSA9IHAwWzFdIC0gNCBpZiBwMFsxXSAtIHRodCAtIDggPiAwIGVsc2UgcDFbMV0gKyB0aHQgKyA2CiAgICAgICAgY3YyLnJlY3RhbmdsZSh2aXMsIChwMFswXSwgdHkgLSB0aHQgLSA0KSwgKHAwWzBdICsgdHcgKyA0LCB0eSArIDMpLCBjb2wsIC0xKQogICAgICAgIGN2Mi5wdXRUZXh0KHZpcywgdHh0LCAocDBbMF0gKyAyLCB0eSksIGN2Mi5GT05UX0hFUlNIRVlfU0lNUExFWCwgZnMsICgyNTUsIDI1NSwgMjU1KSwgbWF4KDEsIHRoIC8vIDIpLCBjdjIuTElORV9BQSkKICAgIGlmIGd0OgogICAgICAgIGN2Mi5wdXRUZXh0KHZpcywgImdyZWVuID0gdHJ1ZSBkZWZlY3QgKGxhYmVsKSIsICg4LCAxMiArIDE0ICogdGgpLCBjdjIuRk9OVF9IRVJTSEVZX1NJTVBMRVgsIGZzLCBHVCwKICAgICAgICAgICAgICAgICAgICBtYXgoMSwgdGggLy8gMiksIGN2Mi5MSU5FX0FBKQogICAgcmV0dXJuIHZpcwoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS13ZWlnaHRzIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zb3VyY2UiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJpbWFnZSBmaWxlIG9yIGZvbGRlciIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0icHJlZGljdGlvbnMiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmYiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PS0xLCBoZWxwPSJvbmUgdGhyZXNob2xkIGZvciBhbGwgY2xhc3NlcyAoZGVmYXVsdDogdGhyZXNob2xkcy5qc29uKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGhyZXNob2xkcyIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iZGVmYXVsdDogdGhyZXNob2xkcy5qc29uIG5leHQgdG8gdGhlIHdlaWdodHMiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWlvdSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC41KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWltZ3N6IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCwgaGVscD0iMCA9IHRoZSB0cmFpbmluZyBpbWFnZSBzaXplIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10aWxlIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4MCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdmVybGFwIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjI1KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRpbGUtYWJvdmUiLCB0eXBlPWludCwgZGVmYXVsdD0yMDAwLCBoZWxwPSJ0aWxlIHBob3RvcyB3aG9zZSBsb25nIHNpZGUgaXMgbGFyZ2VyIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGFiZWxzIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJmb2xkZXIgd2l0aCBZT0xPIGxhYmVsIGZpbGVzOiBkcmF3IHRoZSB0cnVlIGJveGVzIHRvbyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD1Ob25lKQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoKQogICAgbW9kZWwgPSBZT0xPKGEud2VpZ2h0cykKICAgIG5hbWVzID0gbW9kZWwubmFtZXMKICAgIGlmIG5vdCBhLmltZ3N6OgogICAgICAgIGEuaW1nc3ogPSBpbnQoKGdldGF0dHIobW9kZWwsICJja3B0IiwgTm9uZSkgb3Ige30pLmdldCgidHJhaW5fYXJncyIsIHt9KS5nZXQoImltZ3N6IiwgODAwKSkKICAgIHRocl9maWxlID0gUGF0aChhLnRocmVzaG9sZHMpIGlmIGEudGhyZXNob2xkcyBlbHNlIFBhdGgoYS53ZWlnaHRzKS53aXRoX25hbWUoInRocmVzaG9sZHMuanNvbiIpCiAgICBpZiBhLmNvbmYgPj0gMDoKICAgICAgICB0aHIgPSB7aTogYS5jb25mIGZvciBpIGluIG5hbWVzfQogICAgZWxpZiB0aHJfZmlsZS5leGlzdHMoKToKICAgICAgICBwZXIgPSBqc29uLmxvYWRzKHRocl9maWxlLnJlYWRfdGV4dChlbmNvZGluZz0idXRmOCIpKVsicGVyX2NsYXNzIl0KICAgICAgICB0aHIgPSB7aTogZmxvYXQocGVyLmdldChuLCAwLjI1KSkgZm9yIGksIG4gaW4gbmFtZXMuaXRlbXMoKX0KICAgICAgICBwcmludCgicGVyLWNsYXNzIHRocmVzaG9sZHM6Iiwge25hbWVzW2ldOiByb3VuZCh0LCAyKSBmb3IgaSwgdCBpbiB0aHIuaXRlbXMoKX0pCiAgICBlbHNlOgogICAgICAgIHRociA9IHtpOiAwLjI1IGZvciBpIGluIG5hbWVzfQogICAgc3JjID0gUGF0aChhLnNvdXJjZSkKICAgIGZpbGVzID0gc29ydGVkKHAgZm9yIHAgaW4gKHNyYy5pdGVyZGlyKCkgaWYgc3JjLmlzX2RpcigpIGVsc2UgW3NyY10pCiAgICAgICAgICAgICAgICAgICBpZiBwLnN1ZmZpeC5sb3dlcigpIGluIHsiLmpwZyIsICIuanBlZyIsICIucG5nIiwgIi5ibXAiLCAiLnRpZiIsICIudGlmZiIsICIud2VicCJ9KQogICAgb3V0ID0gUGF0aChhLm91dCkKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBzdW1tYXJ5ID0gW10KICAgIGZvciBwIGluIGZpbGVzOgogICAgICAgIGltZyA9IGN2Mi5pbXJlYWQoc3RyKHApKQogICAgICAgIGlmIGltZyBpcyBOb25lOgogICAgICAgICAgICBwcmludCgiY2Fubm90IHJlYWQiLCBwKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGRldHMgPSBbZCBmb3IgZCBpbiBwcmVkaWN0X2ltYWdlKG1vZGVsLCBpbWcsIGEsIG1pbih0aHIudmFsdWVzKCkpKSBpZiBkWzFdID49IHRocltkWzBdXV0KICAgICAgICBndCA9IHJlYWRfZ3QoUGF0aChhLmxhYmVscykgLyAocC5zdGVtICsgIi50eHQiKSwgaW1nLnNoYXBlWzFdLCBpbWcuc2hhcGVbMF0pIGlmIGEubGFiZWxzIGVsc2UgW10KICAgICAgICBjdjIuaW13cml0ZShzdHIob3V0IC8gZiJ7cC5zdGVtfV9hbm5vdGF0ZWQuanBnIiksIGRyYXcoaW1nLCBkZXRzLCBndCwgbmFtZXMpKQogICAgICAgIChvdXQgLyBmIntwLnN0ZW19Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoCiAgICAgICAgICAgIFt7ImNsYXNzIjogbmFtZXNbY10sICJjb25mIjogcm91bmQocywgNCksICJib3giOiBbcm91bmQodiwgMSkgZm9yIHYgaW4gKHgwLCB5MCwgeDEsIHkxKV19CiAgICAgICAgICAgICBmb3IgYywgcywgeDAsIHkwLCB4MSwgeTEgaW4gZGV0c10sIGluZGVudD0xKSkKICAgICAgICBjbnQgPSB7bjogc3VtKDEgZm9yIGQgaW4gZGV0cyBpZiBkWzBdID09IGkpIGZvciBpLCBuIGluIG5hbWVzLml0ZW1zKCl9CiAgICAgICAgc3VtbWFyeS5hcHBlbmQoeyJpbWFnZSI6IHAubmFtZSwgImRlZmVjdHMiOiBsZW4oZGV0cyksICoqKHsidHJ1ZV9ib3hlcyI6IGxlbihndCl9IGlmIGEubGFiZWxzIGVsc2Uge30pLCAqKmNudH0pCiAgICAgICAgcHJpbnQoZiJ7cC5uYW1lfToge2xlbihkZXRzKX0gZGVmZWN0cyIgKyAoZiIgKHRydWU6IHtsZW4oZ3QpfSkiIGlmIGEubGFiZWxzIGVsc2UgIiIpICsKICAgICAgICAgICAgICBmIiB7KHtrOiB2IGZvciBrLCB2IGluIGNudC5pdGVtcygpIGlmIHZ9KX0iKQogICAgaWYgc3VtbWFyeToKICAgICAgICB3aXRoIG9wZW4ob3V0IC8gInN1bW1hcnkuY3N2IiwgInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmOCIpIGFzIGY6CiAgICAgICAgICAgIHdyID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1saXN0KHN1bW1hcnlbMF0ua2V5cygpKSkKICAgICAgICAgICAgd3Iud3JpdGVoZWFkZXIoKQogICAgICAgICAgICB3ci53cml0ZXJvd3Moc3VtbWFyeSkKICAgIHByaW50KCJyZXN1bHRzIC0+Iiwgb3V0KQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "phase5/__init__.py": "", "phase5/check_leakage.py": "IiIiUHJvdmUgdGhlIGhlbGQtb3V0IGV2YWx1YXRpb24gaW1hZ2VzIHdlcmUgbmV2ZXIgdHJhaW5lZCBvbiAobm9yIHVzZWQgZm9yIHZhbGlkYXRpb24gLyBtb2RlbCBzZWxlY3Rpb24pLgoKRXZlcnkgdHJhaW4gKyB2YWwgaW1hZ2Ugb2YgYSB0cmFpbmluZyBzcGVjIChkYXRhLnlhbWwgLyBsaXN0IC50eHQgLyBpbWFnZXMgZGlyKSBpcyBjb21wYXJlZCBhZ2FpbnN0IGV2ZXJ5CmhvbGRvdXQgaW1hZ2Ugb2YgY29udmV5b3JfaG9sZG91dCwgc3hzYzdfaG9sZG91dCAoZnJvemVuIG1hbmlmZXN0KSBhbmQgZGVmZWN0c190ZXN0OgpMRUFLID0gaWRlbnRpY2FsIHNoYTEgb3IgZEhhc2ggSGFtbWluZyA8PSB0aHJlc2hvbGQ7IFdBUk5JTkcgPSBzYW1lIFJvYm9mbG93IHNvdXJjZSBzdGVtIG9ubHkuCgogICAgcHkgLTMuMTAgLW0gcGhhc2U1LmNoZWNrX2xlYWthZ2UgLS1jb25maWcgcGhhc2U1X2NvbmZpZy55YW1sIC0tdHJhaW4gZGF0YS55YW1sIFstLXRyYWluIG1vcmUudHh0XQogICAgICAgIFstLXdyaXRlLWNsZWFuIGNsZWFuX3RyYWluLnR4dF0gWy0tb3V0LWRpciBESVJdICAgICAgICAgICMgZXhpdCBjb2RlIDEgaWYgYW55IExFQUsKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgb3MKaW1wb3J0IGxvZ2dpbmcKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHlhbWwKCmZyb20gcGhhc2U1IGltcG9ydCBjb21tb24KZnJvbSBwaGFzZTUgaW1wb3J0IG1ha2VfaG9sZG91dHMKCmxvZyA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJwaGFzZTUuY2hlY2tfbGVha2FnZSIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBjb2xsZWN0aW5nIHRyYWluIGltYWdlcwoKCmRlZiBfbm9ybShwKSAtPiBzdHI6CiAgICByZXR1cm4gb3MucGF0aC5ub3JtY2FzZShvcy5wYXRoLm5vcm1wYXRoKHN0cihQYXRoKHApLnJlc29sdmUoKSkpKQoKCmRlZiBfZ2xvYl9pbWFnZXMoZDogUGF0aCkgLT4gbGlzdFtzdHJdOgogICAgcmV0dXJuIHNvcnRlZChzdHIocC5yZXNvbHZlKCkpIGZvciBwIGluIGQucmdsb2IoIioiKSBpZiBwLmlzX2ZpbGUoKSBhbmQgcC5zdWZmaXgubG93ZXIoKSBpbiBjb21tb24uSU1HX0VYVFMpCgoKZGVmIF9yZXNvbHZlX2VudHJ5KGJhc2U6IFBhdGgsIGVudHJ5OiBzdHIpIC0+IGxpc3Rbc3RyXToKICAgIHggPSAoYmFzZSAvIGVudHJ5KS5yZXNvbHZlKCkKICAgIGlmIG5vdCB4LmV4aXN0cygpIGFuZCBzdHIoZW50cnkpLnN0YXJ0c3dpdGgoIi4uLyIpOgogICAgICAgIHggPSAoYmFzZSAvIHN0cihlbnRyeSlbMzpdKS5yZXNvbHZlKCkgICMgUm9ib2Zsb3cgZXhwb3J0IHF1aXJrICgnLi4vdHJhaW4vaW1hZ2VzJykKICAgIGlmIHguaXNfZGlyKCk6CiAgICAgICAgcmV0dXJuIF9nbG9iX2ltYWdlcyh4KQogICAgaWYgeC5pc19maWxlKCkgYW5kIHguc3VmZml4Lmxvd2VyKCkgPT0gIi50eHQiOgogICAgICAgIHJldHVybiBjb21tb24ucmVhZF9saXN0KHgpCiAgICBpZiB4LmlzX2ZpbGUoKSBhbmQgeC5zdWZmaXgubG93ZXIoKSBpbiBjb21tb24uSU1HX0VYVFM6CiAgICAgICAgcmV0dXJuIFtzdHIoeCldCiAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmInRyYWluaW5nIGVudHJ5ICd7ZW50cnl9JyBub3QgZm91bmQgKHJlc29sdmVkIHRvIHt4fSkiKQoKCmRlZiB5YW1sX2VudHJpZXMoeWFtbF9wYXRoOiBzdHIgfCBQYXRoLCBrZXlzPSgidHJhaW4iLCAidmFsIikpIC0+IGRpY3Rbc3RyLCBsaXN0W3N0cl1dOgogICAgIiIiSW1hZ2VzIHBlciBrZXkgb2YgYSBkYXRhLnlhbWwsIHJlc29sdmVkIGV4YWN0bHkgbGlrZSBVbHRyYWx5dGljcyAoYmFzZSA9ICdwYXRoJyBrZXkgb3IgeWFtbCBmb2xkZXIpLiIiIgogICAgeXAgPSBQYXRoKHlhbWxfcGF0aCkucmVzb2x2ZSgpCiAgICB3aXRoIG9wZW4oeXAsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZCA9IHlhbWwuc2FmZV9sb2FkKGYpIG9yIHt9CiAgICBiYXNlID0geXAucGFyZW50CiAgICBpZiBkLmdldCgicGF0aCIpOgogICAgICAgIHAgPSBQYXRoKHN0cihkWyJwYXRoIl0pKQogICAgICAgIGJhc2UgPSBwIGlmIHAuaXNfYWJzb2x1dGUoKSBlbHNlICh5cC5wYXJlbnQgLyBwKS5yZXNvbHZlKCkKICAgIG91dCA9IHt9CiAgICBmb3IgayBpbiBrZXlzOgogICAgICAgIHYgPSBkLmdldChrKQogICAgICAgIGlmIG5vdCB2OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGVudHMgPSB2IGlmIGlzaW5zdGFuY2UodiwgKGxpc3QsIHR1cGxlKSkgZWxzZSBbdl0KICAgICAgICBpbWdzID0gW10KICAgICAgICBmb3IgZSBpbiBlbnRzOgogICAgICAgICAgICBpbWdzLmV4dGVuZChfcmVzb2x2ZV9lbnRyeShiYXNlLCBzdHIoZSkpKQogICAgICAgIG91dFtrXSA9IGltZ3MKICAgIHJldHVybiBvdXQKCgpkZWYgY29sbGVjdF90cmFpbl9pbWFnZXMoc3BlYzogc3RyKSAtPiBsaXN0W3N0cl06CiAgICBwID0gUGF0aChzcGVjKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBwLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYidHJhaW5pbmcgc3BlYyBub3QgZm91bmQ6IHtwfSIpCiAgICBpZiBwLmlzX2RpcigpOgogICAgICAgIGltZ3MgPSBfZ2xvYl9pbWFnZXMocCkKICAgIGVsaWYgcC5zdWZmaXgubG93ZXIoKSBpbiAoIi55YW1sIiwgIi55bWwiKToKICAgICAgICBpbWdzID0gW2kgZm9yIHYgaW4geWFtbF9lbnRyaWVzKHAsICgidHJhaW4iLCAidmFsIikpLnZhbHVlcygpIGZvciBpIGluIHZdCiAgICBlbGlmIHAuc3VmZml4Lmxvd2VyKCkgPT0gIi50eHQiOgogICAgICAgIGltZ3MgPSBjb21tb24ucmVhZF9saXN0KHApCiAgICBlbHNlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bnN1cHBvcnRlZCB0cmFpbmluZyBzcGVjICh3YW50IGRhdGEueWFtbCwgbGlzdCAudHh0IG9yIGltYWdlcyBkaXIpOiB7cH0iKQogICAgc2Vlbiwgb3V0ID0gc2V0KCksIFtdCiAgICBmb3IgaSBpbiBpbWdzOgogICAgICAgIGkgPSBzdHIoUGF0aChpKS5yZXNvbHZlKCkpCiAgICAgICAgaWYgaSBub3QgaW4gc2VlbjoKICAgICAgICAgICAgc2Vlbi5hZGQoaSkKICAgICAgICAgICAgb3V0LmFwcGVuZChpKQogICAgcmV0dXJuIG91dAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY2hlY2tpbmcKCgpkZWYgX2hvbGRvdXRfc2V0cyhjZmc6IGRpY3QpIC0+IHR1cGxlW2RpY3Rbc3RyLCBsaXN0W3N0cl1dLCBsaXN0W3N0cl1dOgogICAgd2FybnMgPSBbXQogICAgbWFuID0gbWFrZV9ob2xkb3V0cy5sb2FkX21hbmlmZXN0KGNmZykKICAgIHNldHMgPSB7CiAgICAgICAgImNvbnZleW9yX2hvbGRvdXQiOiBbc3RyKGNvbW1vbi5yZXNvbHZlX3Byb2plY3RfcGF0aChlWyJwYXRoIl0pKSBmb3IgZSBpbiBtYW5bImNvbnZleW9yIl1bImhvbGRvdXQiXV0sCiAgICAgICAgInN4c2M3X2hvbGRvdXQiOiBbc3RyKGNvbW1vbi5yZXNvbHZlX3Byb2plY3RfcGF0aChlWyJwYXRoIl0pKSBmb3IgZSBpbiBtYW5bInN4c2M3Il1bImhvbGRvdXQiXV0sCiAgICB9CiAgICBkID0gY2ZnWyJkYXRhc2V0cyJdWyJkZWZlY3RzIl0KICAgIHRyeToKICAgICAgICBkcyA9IGNvbW1vbi5sb2FkX2RhdGFzZXQoZFsicm9vdCJdLCAiZGVmZWN0cyIpCiAgICAgICAgcmVjcyA9IGRzWyJzcGxpdHMiXS5nZXQoZC5nZXQoImV2YWxfc3BsaXQiLCAidGVzdCIpLCBbXSkKICAgICAgICBpZiBub3QgcmVjczoKICAgICAgICAgICAgd2FybnMuYXBwZW5kKGYiZGVmZWN0czogc3BsaXQgJ3tkLmdldCgnZXZhbF9zcGxpdCcsICd0ZXN0Jyl9JyBub3QgZm91bmQgLSBkZWZlY3RzX3Rlc3Qgbm90IGNoZWNrZWQiKQogICAgICAgIHNldHNbImRlZmVjdHNfdGVzdCJdID0gW3IuaW1hZ2UgZm9yIHIgaW4gcmVjc10KICAgIGV4Y2VwdCBGaWxlTm90Rm91bmRFcnJvciBhcyBlOgogICAgICAgIHdhcm5zLmFwcGVuZChmImRlZmVjdHNfdGVzdCBub3QgY2hlY2tlZDoge2V9IikKICAgICAgICBzZXRzWyJkZWZlY3RzX3Rlc3QiXSA9IFtdCiAgICByZXR1cm4gc2V0cywgd2FybnMKCgpkZWYgX2hhc2hlcyhwYXRoczogbGlzdFtzdHJdKSAtPiB0dXBsZVtsaXN0W3N0cl0sIGxpc3RbaW50IHwgTm9uZV1dOgogICAgc2gsIGRoID0gW10sIFtdCiAgICBmb3IgcCBpbiBwYXRoczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNoLmFwcGVuZChjb21tb24uZmlsZV9zaGExKHApKQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICBzaC5hcHBlbmQoIiIpCiAgICAgICAgaW1nID0gY29tbW9uLmltcmVhZChwKQogICAgICAgIGRoLmFwcGVuZChOb25lIGlmIGltZyBpcyBOb25lIGVsc2UgY29tbW9uLmRoYXNoKGltZykpCiAgICByZXR1cm4gc2gsIGRoCgoKZGVmIGNoZWNrX2xlYWthZ2UoY2ZnOiBkaWN0LCB0cmFpbl9pbWFnZXM6IGxpc3Rbc3RyXSwgbmVhcl9kdXBfaGFtbWluZzogaW50IHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Q6CiAgICB0aHIgPSBpbnQoY2ZnLmdldCgibmVhcl9kdXBfaGFtbWluZyIsIDYpIGlmIG5lYXJfZHVwX2hhbW1pbmcgaXMgTm9uZSBlbHNlIG5lYXJfZHVwX2hhbW1pbmcpCiAgICBzZXRzLCB3YXJuaW5ncyA9IF9ob2xkb3V0X3NldHMoY2ZnKQogICAgdHJhaW5faW1hZ2VzID0gW3N0cihQYXRoKHApLnJlc29sdmUoKSkgZm9yIHAgaW4gdHJhaW5faW1hZ2VzXQogICAgdF9zaGEsIHRfZGggPSBfaGFzaGVzKHRyYWluX2ltYWdlcykKICAgIGZvciBwLCBzLCBoIGluIHppcCh0cmFpbl9pbWFnZXMsIHRfc2hhLCB0X2RoKToKICAgICAgICBpZiBub3QgczoKICAgICAgICAgICAgd2FybmluZ3MuYXBwZW5kKGYidHJhaW4gaW1hZ2UgbWlzc2luZzoge3B9IikKICAgICAgICBlbGlmIGggaXMgTm9uZToKICAgICAgICAgICAgd2FybmluZ3MuYXBwZW5kKGYidHJhaW4gaW1hZ2UgdW5yZWFkYWJsZSAoc2hhMSBjaGVjayBvbmx5KToge3B9IikKICAgIHRfc3RlbSA9IFtjb21tb24uc291cmNlX3N0ZW0ocCkgZm9yIHAgaW4gdHJhaW5faW1hZ2VzXQogICAgbGVha3MsIG1pbmggPSBbXSwge30KICAgIGZvciBuYW1lLCBob2xkIGluIHNldHMuaXRlbXMoKToKICAgICAgICBpZiBub3QgaG9sZDoKICAgICAgICAgICAgbWluaFtuYW1lXSA9IE5vbmUKICAgICAgICAgICAgY29udGludWUKICAgICAgICBoX3NoYSwgaF9kaCA9IF9oYXNoZXMoaG9sZCkKICAgICAgICBzaGFfaWR4ID0ge30KICAgICAgICBmb3IgaiwgcyBpbiBlbnVtZXJhdGUoaF9zaGEpOgogICAgICAgICAgICBzaGFfaWR4LnNldGRlZmF1bHQocywgaikKICAgICAgICBzdGVtX2lkeCA9IHt9CiAgICAgICAgZm9yIGosIHAgaW4gZW51bWVyYXRlKGhvbGQpOgogICAgICAgICAgICBzdGVtX2lkeC5zZXRkZWZhdWx0KGNvbW1vbi5zb3VyY2Vfc3RlbShwKSwgaikKICAgICAgICB0aSA9IFtpIGZvciBpLCBoIGluIGVudW1lcmF0ZSh0X2RoKSBpZiBoIGlzIG5vdCBOb25lXQogICAgICAgIGhqID0gW2ogZm9yIGosIGggaW4gZW51bWVyYXRlKGhfZGgpIGlmIGggaXMgbm90IE5vbmVdCiAgICAgICAgYmVzdCA9IHt9CiAgICAgICAgaWYgdGkgYW5kIGhqOgogICAgICAgICAgICBtID0gY29tbW9uLmhhbW1pbmdfbWF0cml4KFt0X2RoW2ldIGZvciBpIGluIHRpXSwgW2hfZGhbal0gZm9yIGogaW4gaGpdKQogICAgICAgICAgICBtaW5oW25hbWVdID0gaW50KG0ubWluKCkpCiAgICAgICAgICAgIGFtID0gbS5hcmdtaW4oMSkKICAgICAgICAgICAgZm9yIHIsIGkgaW4gZW51bWVyYXRlKHRpKToKICAgICAgICAgICAgICAgIGJlc3RbaV0gPSAoaW50KG1bciwgYW1bcl1dKSwgaGpbaW50KGFtW3JdKV0pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbWluaFtuYW1lXSA9IE5vbmUKICAgICAgICBmb3IgaSwgcCBpbiBlbnVtZXJhdGUodHJhaW5faW1hZ2VzKToKICAgICAgICAgICAgaWYgdF9zaGFbaV0gYW5kIHRfc2hhW2ldIGluIHNoYV9pZHg6CiAgICAgICAgICAgICAgICBqID0gc2hhX2lkeFt0X3NoYVtpXV0KICAgICAgICAgICAgICAgIGxlYWtzLmFwcGVuZCh7InRyYWluIjogcCwgImhvbGRvdXQiOiBob2xkW2pdLCAiZXZhbF9zZXQiOiBuYW1lLCAicmVhc29uIjogImlkZW50aWNhbF9zaGExIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImhhbW1pbmciOiAwfSkKICAgICAgICAgICAgZWxpZiBpIGluIGJlc3QgYW5kIGJlc3RbaV1bMF0gPD0gdGhyOgogICAgICAgICAgICAgICAgbGVha3MuYXBwZW5kKHsidHJhaW4iOiBwLCAiaG9sZG91dCI6IGhvbGRbYmVzdFtpXVsxXV0sICJldmFsX3NldCI6IG5hbWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJyZWFzb24iOiAibmVhcl9kdXBsaWNhdGVfZGhhc2giLCAiaGFtbWluZyI6IGJlc3RbaV1bMF19KQogICAgICAgICAgICBlbGlmIHRfc3RlbVtpXSBpbiBzdGVtX2lkeDoKICAgICAgICAgICAgICAgIHdhcm5pbmdzLmFwcGVuZCh7InRyYWluIjogcCwgImhvbGRvdXQiOiBob2xkW3N0ZW1faWR4W3Rfc3RlbVtpXV1dLCAiZXZhbF9zZXQiOiBuYW1lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicmVhc29uIjogInNhbWVfc291cmNlX3N0ZW0ifSkKICAgIHJldHVybiB7Im9rIjogbm90IGxlYWtzLCAibl90cmFpbiI6IGxlbih0cmFpbl9pbWFnZXMpLCAibmVhcl9kdXBfaGFtbWluZyI6IHRociwgImxlYWtzIjogbGVha3MsCiAgICAgICAgICAgICJ3YXJuaW5ncyI6IHdhcm5pbmdzLCAibWluX2hhbW1pbmdfcGVyX3NldCI6IG1pbmgsCiAgICAgICAgICAgICJuX2hvbGRvdXRfcGVyX3NldCI6IHtrOiBsZW4odikgZm9yIGssIHYgaW4gc2V0cy5pdGVtcygpfX0KCgpkZWYgbWFpbihhcmd2PU5vbmUpIC0+IGludDoKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXy5zcGxpdGxpbmVzKClbMF0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRyYWluIiwgYWN0aW9uPSJhcHBlbmQiLCByZXF1aXJlZD1UcnVlLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImRhdGEueWFtbCAvIGxpc3QgLnR4dCAvIGltYWdlcyBkaXIgKHJlcGVhdGFibGUpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS13cml0ZS1jbGVhbiIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0id3JpdGUgdGhlIHRyYWluIGxpc3Qgd2l0aCBsZWFrZWQgaW1hZ2VzIHJlbW92ZWQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW5lYXItZHVwLWhhbW1pbmciLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dC1kaXIiLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImRlZmF1bHQ6IGNvbmZpZyBvdXRfZGlyIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKGFyZ3YpCiAgICBsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGxldmVsbmFtZSlzICUobWVzc2FnZSlzIikKICAgIGNmZyA9IGNvbW1vbi5sb2FkX2NvbmZpZyhhLmNvbmZpZykKICAgIHRyeToKICAgICAgICBpbWdzLCBzZWVuID0gW10sIHNldCgpCiAgICAgICAgZm9yIHNwZWMgaW4gYS50cmFpbjoKICAgICAgICAgICAgZm9yIHAgaW4gY29sbGVjdF90cmFpbl9pbWFnZXMoc3BlYyk6CiAgICAgICAgICAgICAgICBpZiBwIG5vdCBpbiBzZWVuOgogICAgICAgICAgICAgICAgICAgIHNlZW4uYWRkKHApCiAgICAgICAgICAgICAgICAgICAgaW1ncy5hcHBlbmQocCkKICAgICAgICByZXAgPSBjaGVja19sZWFrYWdlKGNmZywgaW1ncywgYS5uZWFyX2R1cF9oYW1taW5nKQogICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvciwgVmFsdWVFcnJvcikgYXMgZToKICAgICAgICBsb2cuZXJyb3IoIiVzIiwgZSkKICAgICAgICByZXR1cm4gMgogICAgIyB5YW1sICd0ZXN0JyBwb2ludGluZyBhdCBob2xkb3V0IGltYWdlcyBpcyBub3QgdHJhaW5pbmcsIGJ1dCB3b3J0aCBmbGFnZ2luZwogICAgaG9sZF9hbGwgPSBzZXQoKQogICAgZm9yIGsgaW4gbWFrZV9ob2xkb3V0cy5IT0xET1VUX0tFWVM6CiAgICAgICAgaG9sZF9hbGwgfD0ge19ub3JtKGNvbW1vbi5yZXNvbHZlX3Byb2plY3RfcGF0aChlWyJwYXRoIl0pKSBmb3IgZSBpbiBtYWtlX2hvbGRvdXRzLmxvYWRfbWFuaWZlc3QoY2ZnKVtrXVsiaG9sZG91dCJdfQogICAgZm9yIHNwZWMgaW4gYS50cmFpbjoKICAgICAgICBpZiBQYXRoKHNwZWMpLnN1ZmZpeC5sb3dlcigpIGluICgiLnlhbWwiLCAiLnltbCIpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB0c3QgPSB5YW1sX2VudHJpZXMoc3BlYywgKCJ0ZXN0IiwpKS5nZXQoInRlc3QiLCBbXSkKICAgICAgICAgICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOgogICAgICAgICAgICAgICAgdHN0ID0gW10KICAgICAgICAgICAgbiA9IGxlbih7X25vcm0ocCkgZm9yIHAgaW4gdHN0fSAmIGhvbGRfYWxsKQogICAgICAgICAgICBpZiBuOgogICAgICAgICAgICAgICAgcmVwWyJ3YXJuaW5ncyJdLmFwcGVuZChmIntzcGVjfTogJ3Rlc3QnIGVudHJ5IHBvaW50cyBhdCB7bn0gaG9sZG91dCBpbWFnZXMgKG9rIG9ubHkgZm9yIHRlc3RpbmcpIikKICAgIG91dF9kaXIgPSBQYXRoKGEub3V0X2RpciBvciBjZmdbIm91dF9kaXIiXSkKICAgIGNvbW1vbi5kdW1wX2pzb24ob3V0X2RpciAvICJsZWFrYWdlX3JlcG9ydC5qc29uIiwgcmVwKQogICAgaWYgYS53cml0ZV9jbGVhbjoKICAgICAgICBiYWQgPSB7bFsidHJhaW4iXSBmb3IgbCBpbiByZXBbImxlYWtzIl19CiAgICAgICAgY29tbW9uLndyaXRlX2xpc3QoYS53cml0ZV9jbGVhbiwgW3AgZm9yIHAgaW4gaW1ncyBpZiBwIG5vdCBpbiBiYWRdKQogICAgICAgIHByaW50KGYiY2xlYW4gbGlzdDoge2Eud3JpdGVfY2xlYW59ICh7bGVuKGltZ3MpIC0gbGVuKGJhZCl9IG9mIHtsZW4oaW1ncyl9IGltYWdlcykiKQogICAgcHJpbnQoZiJsZWFrYWdlIGNoZWNrOiB7J09LJyBpZiByZXBbJ29rJ10gZWxzZSAnTEVBSyd9IC0ge2xlbihyZXBbJ2xlYWtzJ10pfSBsZWFrcywgIgogICAgICAgICAgZiJ7bGVuKHJlcFsnd2FybmluZ3MnXSl9IHdhcm5pbmdzLCBuX3RyYWluPXtyZXBbJ25fdHJhaW4nXX07IHJlcG9ydCB7b3V0X2RpciAvICdsZWFrYWdlX3JlcG9ydC5qc29uJ30iKQogICAgZm9yIGwgaW4gcmVwWyJsZWFrcyJdWzoyMF06CiAgICAgICAgcHJpbnQoZiIgIExFQUsgW3tsWydldmFsX3NldCddfV0ge2xbJ3JlYXNvbiddfSAoaD17bFsnaGFtbWluZyddfSk6IHtsWyd0cmFpbiddfSB+IHtsWydob2xkb3V0J119IikKICAgIHJldHVybiAwIGlmIHJlcFsib2siXSBlbHNlIDEKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==", "phase5/common.py": "...", "phase5/evaluate_class_agnostic.py": "...", "phase5/failure_analysis.py": "...", "phase5/make_holdouts.py": "...", "phase5/metrics.py": "IiIiQ09DTy1mYWl0aGZ1bCBkZXRlY3Rpb24gbWV0cmljcyBpbiBwdXJlIG51bXB5IChubyBJTykuCgpgZXZhbHVhdGVgIHJlcHJvZHVjZXMgcHljb2NvdG9vbHMgQ09DT2V2YWwgYmJveCBBUCAoaXNjcm93ZD0wLCBvbmUgY2F0ZWdvcnksIG1heERldHM9W21heF9kZXRdLAphcmVhUm5nPVthcmVhX3JuZ10pIHRvIGZsb2F0aW5nLXBvaW50IHByZWNpc2lvbiwgYW5kIGFkZHMgUC9SL0YxIGF0IGFuIG9wZXJhdGluZyBjb25maWRlbmNlLCBhIGJlc3QtRjEKc2NhbiwgbWF4IHJlY2FsbCBhbmQgdGhlIHJhdyBQUiBjdXJ2ZS4gYGV2YWx1YXRlX3NpemVzYCBydW5zIGl0IGZvciBhbGwvc21hbGwvbWVkaXVtL2xhcmdlOwpgbWF0Y2hfZGV0YWlsc2AgZ2l2ZXMgcGVyLWJveCBtYXRjaCBzdGF0dXMgZm9yIG9uZSBpbWFnZSAoZmFpbHVyZSBhbmFseXNpcykuCgpDb252ZW50aW9ucyAoc2VlIHBoYXNlNS9jb21tb24ucHkpOiBHVCAoTiwgNSkgPSBbY2xzLCB4MSwgeTEsIHgyLCB5Ml07IHByZWRzIChNLCA2KSA9IFt4MSwgeTEsIHgyLCB5MiwgY29uZiwgY2xzXS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBtYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLiBpbXBvcnQgY29tbW9uCgpfUkVDX1RIUlMgPSBucC5saW5zcGFjZSgwLjAsIDEuMCwgMTAxKQpfTUFYX1BSX1BPSU5UUyA9IDEwMDAKCgpkZWYgYm94X2lvdShhOiBucC5uZGFycmF5LCBiOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgIiIiUGFpcndpc2UgSW9VIG9mIHh5eHkgYm94ZXMgKE4sIDQpIHggKE0sIDQpIC0+IChOLCBNKS4gU2FtZSBmb3JtdWxhIGFzIHB5Y29jb3Rvb2xzIChubyArMSkuIiIiCiAgICBhID0gbnAuYXNhcnJheShhLCBkdHlwZT1ucC5mbG9hdDY0KS5yZXNoYXBlKC0xLCA0KQogICAgYiA9IG5wLmFzYXJyYXkoYiwgZHR5cGU9bnAuZmxvYXQ2NCkucmVzaGFwZSgtMSwgNCkKICAgIGlmIGxlbihhKSA9PSAwIG9yIGxlbihiKSA9PSAwOgogICAgICAgIHJldHVybiBucC56ZXJvcygobGVuKGEpLCBsZW4oYikpKQogICAgYXcgPSAoYVs6LCAyXSAtIGFbOiwgMF0pLmNsaXAoMCkKICAgIGFoID0gKGFbOiwgM10gLSBhWzosIDFdKS5jbGlwKDApCiAgICBidyA9IChiWzosIDJdIC0gYls6LCAwXSkuY2xpcCgwKQogICAgYmggPSAoYls6LCAzXSAtIGJbOiwgMV0pLmNsaXAoMCkKICAgIGl3ID0gKG5wLm1pbmltdW0oYVs6LCBOb25lLCAyXSwgYltOb25lLCA6LCAyXSkgLSBucC5tYXhpbXVtKGFbOiwgTm9uZSwgMF0sIGJbTm9uZSwgOiwgMF0pKS5jbGlwKDApCiAgICBpaCA9IChucC5taW5pbXVtKGFbOiwgTm9uZSwgM10sIGJbTm9uZSwgOiwgM10pIC0gbnAubWF4aW11bShhWzosIE5vbmUsIDFdLCBiW05vbmUsIDosIDFdKSkuY2xpcCgwKQogICAgaW50ZXIgPSBpdyAqIGloCiAgICB1bmlvbiA9IChhdyAqIGFoKVs6LCBOb25lXSArIChidyAqIGJoKVtOb25lLCA6XSAtIGludGVyCiAgICB3aXRoIG5wLmVycnN0YXRlKGRpdmlkZT0iaWdub3JlIiwgaW52YWxpZD0iaWdub3JlIik6CiAgICAgICAgaW91ID0gbnAud2hlcmUodW5pb24gPiAwLCBpbnRlciAvIHVuaW9uLCAwLjApCiAgICByZXR1cm4gaW91CgoKZGVmIF9sYXN0X2FyZ21heCh2OiBucC5uZGFycmF5KSAtPiBpbnQ6CiAgICAiIiJJbmRleCBvZiB0aGUgbWF4aW11bSwgdGllcyByZXNvbHZlZCB0byB0aGUgTEFTVCBvY2N1cnJlbmNlIChDT0NPJ3MgYGlvdXMgPCBpb3U6IGNvbnRpbnVlYCBydWxlKS4iIiIKICAgIHJldHVybiBpbnQobGVuKHYpIC0gMSAtIG5wLmFyZ21heCh2Wzo6LTFdKSkKCgpkZWYgX2NhbmRpZGF0ZXMoaW91czogbnAubmRhcnJheSwgdG1pbjogZmxvYXQpOgogICAgIiIiU3BhcnNlIGNhbmRpZGF0ZSBsaXN0cyBjb21wdXRlZCBvbmNlIHBlciBpbWFnZTogZm9yIGV2ZXJ5IGRldCB3aG9zZSBiZXN0IElvVSA+PSB0bWluLCB0aGUgR1QKICAgIGNvbHVtbnMgd2l0aCBJb1UgPj0gdG1pbiBhbmQgdGhlaXIgSW9VcyAocGxhaW4gUHl0aG9uIGxpc3RzLCBhc2NlbmRpbmcgY29sdW1uIG9yZGVyKS4iIiIKICAgIGlmIGlvdXMuc2l6ZSA9PSAwOgogICAgICAgIHJldHVybiBbXQogICAgcm93cyA9IG5wLm5vbnplcm8oaW91cy5tYXgoYXhpcz0xKSA+PSB0bWluKVswXQogICAgb3V0ID0gW10KICAgIGZvciBkIGluIHJvd3MudG9saXN0KCk6CiAgICAgICAgY29scyA9IG5wLm5vbnplcm8oaW91c1tkXSA+PSB0bWluKVswXQogICAgICAgIG91dC5hcHBlbmQoKGQsIGNvbHMudG9saXN0KCksIGlvdXNbZCwgY29sc10udG9saXN0KCkpKQogICAgcmV0dXJuIG91dAoKCmRlZiBfY29jb19tYXRjaChpb3VzOiBucC5uZGFycmF5LCBndF9pZzogbnAubmRhcnJheSwgdGhyOiBmbG9hdCwgY2FuZHM9Tm9uZSk6CiAgICAiIiJDT0NPIGV2YWx1YXRlSW1nIGdyZWVkeSBtYXRjaGluZyBmb3Igb25lIHRocmVzaG9sZC4gRGV0cyBtdXN0IGJlIGluIGNvbmYgb3JkZXIuIFJldHVybnMKICAgIGR0X21hdGNoZWRfZ3RfaW5kZXggKEQsKSBpbnQsIC0xID0gdW5tYXRjaGVkLiBBbW9uZyB1bnRha2VuIEdUIHdpdGggSW9VID49IHQsIG5vbi1pZ25vcmVkIEdUIGFyZQogICAgcHJlZmVycmVkOyB3aXRoaW4gYSBncm91cCB0aGUgbWF4IElvVSB3aW5zIHdpdGggdGllcyB0byB0aGUgTEFTVCBpbmRleCAoQ09DTydzIGBpb3VzIDwgaW91YCBydWxlKS4KICAgIERldHMgd2l0aCBubyBHVCBhdCBJb1UgPj0gdCBhcmUgc2tpcHBlZCB3aXRob3V0IHBlci1kZXQgd29yay4gYGNhbmRzYCAoZnJvbSBfY2FuZGlkYXRlcyB3aXRoIGEKICAgIHRocmVzaG9sZCA8PSB0KSBjYW4gYmUgcGFzc2VkIHRvIHNoYXJlIHRoZSBzcGFyc2Ugc3RydWN0dXJlIGFjcm9zcyB0aHJlc2hvbGRzLiIiIgogICAgbmQsIG5nID0gaW91cy5zaGFwZQogICAgdCA9IG1pbih0aHIsIDEgLSAxZS0xMCkKICAgIGR0bSA9IG5wLmZ1bGwobmQsIC0xLCBkdHlwZT1ucC5pbnQ2NCkKICAgIGlmIG5nID09IDAgb3IgbmQgPT0gMDoKICAgICAgICByZXR1cm4gZHRtCiAgICBpZiBjYW5kcyBpcyBOb25lOgogICAgICAgIGNhbmRzID0gX2NhbmRpZGF0ZXMoaW91cywgdCkKICAgIGlnID0gbnAuYXNhcnJheShndF9pZykuYXN0eXBlKGJvb2wpLnRvbGlzdCgpCiAgICB0YWtlbiA9IFtGYWxzZV0gKiBuZwogICAgZm9yIGQsIGNvbHMsIHZhbHMgaW4gY2FuZHM6CiAgICAgICAgYm4sIGJudiwgYmksIGJpdiA9IC0xLCAtMS4wLCAtMSwgLTEuMAogICAgICAgIGZvciBjLCB2IGluIHppcChjb2xzLCB2YWxzKToKICAgICAgICAgICAgaWYgdiA8IHQgb3IgdGFrZW5bY106CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBpZ1tjXToKICAgICAgICAgICAgICAgIGlmIHYgPj0gYml2OgogICAgICAgICAgICAgICAgICAgIGJpLCBiaXYgPSBjLCB2CiAgICAgICAgICAgIGVsaWYgdiA+PSBibnY6CiAgICAgICAgICAgICAgICBibiwgYm52ID0gYywgdgogICAgICAgIG0gPSBibiBpZiBibiA+PSAwIGVsc2UgYmkKICAgICAgICBpZiBtID49IDA6CiAgICAgICAgICAgIHRha2VuW21dID0gVHJ1ZQogICAgICAgICAgICBkdG1bZF0gPSBtCiAgICByZXR1cm4gZHRtCgoKZGVmIF9mKHgpIC0+IGZsb2F0OgogICAgcmV0dXJuIGZsb2F0KHgpCgoKZGVmIF9lbXB0eV9yZXN1bHQobl9pbWFnZXMsIGlvdV90aHJzLCBvcF9jb25mKToKICAgIG5hbiA9IGZsb2F0KCJuYW4iKQogICAgcmV0dXJuIHsKICAgICAgICAibl9pbWFnZXMiOiBuX2ltYWdlcywgIm5fZ3QiOiAwLCAibl9wcmVkIjogMCwgImFwNTAiOiBuYW4sICJhcCI6IG5hbiwKICAgICAgICAiYXBfcGVyX2lvdSI6IHtmbG9hdCh0KTogbmFuIGZvciB0IGluIGlvdV90aHJzfSwgIm9wX2NvbmYiOiBmbG9hdChvcF9jb25mKSwKICAgICAgICAidHAiOiAwLCAiZnAiOiAwLCAiZm4iOiAwLCAicHJlY2lzaW9uIjogbmFuLCAicmVjYWxsIjogbmFuLCAiZjEiOiBuYW4sCiAgICAgICAgImJlc3RfZjEiOiBuYW4sICJiZXN0X2YxX2NvbmYiOiBuYW4sICJwcmVjaXNpb25fYXRfYmVzdF9mMSI6IG5hbiwgInJlY2FsbF9hdF9iZXN0X2YxIjogbmFuLAogICAgICAgICJtYXhfcmVjYWxsIjogbmFuLCAicHJfY3VydmUiOiB7InJlY2FsbCI6IFtdLCAicHJlY2lzaW9uIjogW10sICJjb25mIjogW119LAogICAgfQoKCmRlZiBldmFsdWF0ZShndHMsIHByZWRzLCAqLCBpb3VfdGhycz0oMC41LCksIGFyZWFfcm5nPSgwLjAsIDFlMTApLCBhcmVhX3NjYWxlcz1Ob25lLAogICAgICAgICAgICAgbWF4X2RldDogaW50ID0gMzAwLCBvcF9jb25mOiBmbG9hdCA9IDAuMjUsIGNsYXNzX2lkPU5vbmUpIC0+IGRpY3Q6CiAgICAiIiJDT0NPIGJib3ggZXZhbHVhdGlvbiBvdmVyIGEgbGlzdCBvZiBpbWFnZXMgKHNlZSBDT05UUkFDVC5tZCBmb3IgdGhlIHJldHVybmVkIGtleXMpLiIiIgogICAgaWYgbGVuKGd0cykgIT0gbGVuKHByZWRzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZ3RzICh7bGVuKGd0cyl9KSBhbmQgcHJlZHMgKHtsZW4ocHJlZHMpfSkgZGlmZmVyIGluIGxlbmd0aCIpCiAgICBpb3VfdGhycyA9IFtmbG9hdCh0KSBmb3IgdCBpbiBucC5hdGxlYXN0XzFkKG5wLmFzYXJyYXkoaW91X3RocnMsIGR0eXBlPW5wLmZsb2F0NjQpKV0KICAgIGlmIG5vdCBpb3VfdGhyczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpb3VfdGhycyBpcyBlbXB0eSIpCiAgICBsbywgaGkgPSBmbG9hdChhcmVhX3JuZ1swXSksIGZsb2F0KGFyZWFfcm5nWzFdKQogICAgbl9pbWcgPSBsZW4oZ3RzKQogICAgaWYgYXJlYV9zY2FsZXMgaXMgTm9uZToKICAgICAgICBhcmVhX3NjYWxlcyA9IFsxLjBdICogbl9pbWcKICAgIGlmIGxlbihhcmVhX3NjYWxlcykgIT0gbl9pbWc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYXJlYV9zY2FsZXMgbGVuZ3RoIG11c3QgZXF1YWwgbnVtYmVyIG9mIGltYWdlcyIpCiAgICBudCA9IGxlbihpb3VfdGhycykKCiAgICBzY29yZXNfbCwgZHRtX2wsIGR0aWdfbCwgZ3RpZ19sID0gW10sIFtdLCBbXSwgW10KICAgIGZvciBpIGluIHJhbmdlKG5faW1nKToKICAgICAgICBnID0gbnAuYXNhcnJheShndHNbaV0sIGR0eXBlPW5wLmZsb2F0NjQpLnJlc2hhcGUoLTEsIDUpCiAgICAgICAgcCA9IG5wLmFzYXJyYXkocHJlZHNbaV0sIGR0eXBlPW5wLmZsb2F0NjQpLnJlc2hhcGUoLTEsIDYpCiAgICAgICAgaWYgY2xhc3NfaWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGcgPSBnW2dbOiwgMF0gPT0gY2xhc3NfaWRdCiAgICAgICAgICAgIHAgPSBwW3BbOiwgNV0gPT0gY2xhc3NfaWRdCiAgICAgICAgaWYgbGVuKGcpID09IDAgYW5kIGxlbihwKSA9PSAwOgogICAgICAgICAgICBjb250aW51ZSAgIyBDT0NPOiBldmFsdWF0ZUltZyByZXR1cm5zIE5vbmUKICAgICAgICBzMiA9IGZsb2F0KGFyZWFfc2NhbGVzW2ldKSAqKiAyCiAgICAgICAgZ19hcmVhID0gY29tbW9uLmJveF9hcmVhcyhnWzosIDE6NV0pICogczIKICAgICAgICBnX2lnID0gKChnX2FyZWEgPCBsbykgfCAoZ19hcmVhID4gaGkpKS5hc3R5cGUobnAuaW50NjQpCiAgICAgICAgZ2kgPSBucC5hcmdzb3J0KGdfaWcsIGtpbmQ9Im1lcmdlc29ydCIpCiAgICAgICAgZywgZ19hcmVhLCBnX2lnID0gZ1tnaV0sIGdfYXJlYVtnaV0sIGdfaWdbZ2ldCiAgICAgICAgZGkgPSBucC5hcmdzb3J0KC1wWzosIDRdLCBraW5kPSJtZXJnZXNvcnQiKVs6bWF4X2RldF0KICAgICAgICBwID0gcFtkaV0KICAgICAgICBkX2FyZWEgPSBjb21tb24uYm94X2FyZWFzKHBbOiwgMDo0XSkgKiBzMgogICAgICAgIGRfb3V0ID0gKGRfYXJlYSA8IGxvKSB8IChkX2FyZWEgPiBoaSkKICAgICAgICBpb3VzID0gYm94X2lvdShwWzosIDA6NF0sIGdbOiwgMTo1XSkKICAgICAgICBkdG0gPSBucC56ZXJvcygobnQsIGxlbihwKSksIGR0eXBlPWJvb2wpCiAgICAgICAgZHRpZyA9IG5wLnplcm9zKChudCwgbGVuKHApKSwgZHR5cGU9Ym9vbCkKICAgICAgICBjYW5kcyA9IF9jYW5kaWRhdGVzKGlvdXMsIG1pbihtaW4oaW91X3RocnMpLCAxIC0gMWUtMTApKSBpZiBsZW4oZykgZWxzZSBbXQogICAgICAgIGZvciB0aSwgdCBpbiBlbnVtZXJhdGUoaW91X3RocnMpOgogICAgICAgICAgICBtID0gX2NvY29fbWF0Y2goaW91cywgZ19pZywgdCwgY2FuZHMpCiAgICAgICAgICAgIG1hdGNoZWQgPSBtID49IDAKICAgICAgICAgICAgZHRtW3RpXSA9IG1hdGNoZWQKICAgICAgICAgICAgaWYgbGVuKGcpOgogICAgICAgICAgICAgICAgZHRpZ1t0aV0gPSBucC53aGVyZShtYXRjaGVkLCBnX2lnW25wLm1heGltdW0obSwgMCldLmFzdHlwZShib29sKSwgRmFsc2UpCiAgICAgICAgICAgIGR0aWdbdGldIHw9ICh+bWF0Y2hlZCkgJiBkX291dAogICAgICAgIHNjb3Jlc19sLmFwcGVuZChwWzosIDRdKQogICAgICAgIGR0bV9sLmFwcGVuZChkdG0pCiAgICAgICAgZHRpZ19sLmFwcGVuZChkdGlnKQogICAgICAgIGd0aWdfbC5hcHBlbmQoZ19pZykKCiAgICBpZiBub3QgZ3RpZ19sOgogICAgICAgIHJldHVybiBfZW1wdHlfcmVzdWx0KG5faW1nLCBpb3VfdGhycywgb3BfY29uZikKICAgIHNjb3JlcyA9IG5wLmNvbmNhdGVuYXRlKHNjb3Jlc19sKQogICAgb3JkZXIgPSBucC5hcmdzb3J0KC1zY29yZXMsIGtpbmQ9Im1lcmdlc29ydCIpCiAgICBzY29yZXMgPSBzY29yZXNbb3JkZXJdCiAgICBkdG0gPSBucC5jb25jYXRlbmF0ZShkdG1fbCwgYXhpcz0xKVs6LCBvcmRlcl0KICAgIGR0aWcgPSBucC5jb25jYXRlbmF0ZShkdGlnX2wsIGF4aXM9MSlbOiwgb3JkZXJdCiAgICBndGlnID0gbnAuY29uY2F0ZW5hdGUoZ3RpZ19sKQogICAgbnBpZyA9IGludChucC5jb3VudF9ub256ZXJvKGd0aWcgPT0gMCkpCiAgICBpZiBucGlnID09IDA6CiAgICAgICAgciA9IF9lbXB0eV9yZXN1bHQobl9pbWcsIGlvdV90aHJzLCBvcF9jb25mKQogICAgICAgIHJbIm5fcHJlZCJdID0gaW50KG5wLmNvdW50X25vbnplcm8ofmR0aWdbMF0pKQogICAgICAgIHJbImZwIl0gPSBpbnQobnAuY291bnRfbm9uemVybygofmR0bVswXSkgJiAofmR0aWdbMF0pICYgKHNjb3JlcyA+PSBvcF9jb25mKSkpCiAgICAgICAgclsidHAiXSA9IDAKICAgICAgICByZXR1cm4gcgoKICAgIHRwcyA9IGR0bSAmIH5kdGlnCiAgICBmcHMgPSB+ZHRtICYgfmR0aWcKICAgIHRwX3N1bSA9IG5wLmN1bXN1bSh0cHMsIGF4aXM9MSkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBmcF9zdW0gPSBucC5jdW1zdW0oZnBzLCBheGlzPTEpLmFzdHlwZShucC5mbG9hdDY0KQogICAgYXBfcGVyID0ge30KICAgIHJjMCA9IHByMCA9IE5vbmUKICAgIGZvciB0aSBpbiByYW5nZShudCk6CiAgICAgICAgdHAsIGZwID0gdHBfc3VtW3RpXSwgZnBfc3VtW3RpXQogICAgICAgIG5kID0gbGVuKHRwKQogICAgICAgIHJjID0gdHAgLyBucGlnCiAgICAgICAgcHIgPSB0cCAvIChmcCArIHRwICsgbnAuc3BhY2luZygxKSkKICAgICAgICBpZiB0aSA9PSAwOgogICAgICAgICAgICByYzAsIHByMCA9IHJjLmNvcHkoKSwgcHIuY29weSgpCiAgICAgICAgcSA9IG5wLnplcm9zKGxlbihfUkVDX1RIUlMpKQogICAgICAgIGlmIG5kOgogICAgICAgICAgICBwcl9lbnYgPSBucC5tYXhpbXVtLmFjY3VtdWxhdGUocHJbOjotMV0pWzo6LTFdCiAgICAgICAgICAgIGluZHMgPSBucC5zZWFyY2hzb3J0ZWQocmMsIF9SRUNfVEhSUywgc2lkZT0ibGVmdCIpCiAgICAgICAgICAgIHZhbGlkID0gaW5kcyA8IG5kCiAgICAgICAgICAgIHFbdmFsaWRdID0gcHJfZW52W2luZHNbdmFsaWRdXQogICAgICAgIGFwX3Blcltpb3VfdGhyc1t0aV1dID0gZmxvYXQobnAubWVhbihxKSkKCiAgICAjIC0tLS0gb3BlcmF0aW5nIHBvaW50IC8gYmVzdCBGMSAvIFBSIGN1cnZlIGF0IGlvdV90aHJzWzBdIChub24taWdub3JlZCBkZXRzIG9ubHkpCiAgICBrZWVwID0gfmR0aWdbMF0KICAgIHNfayA9IHNjb3Jlc1trZWVwXQogICAgdHBfayA9IGR0bVswXVtrZWVwXQogICAgbl9wcmVkID0gaW50KGxlbihzX2spKQogICAgc2VsID0gc19rID49IG9wX2NvbmYKICAgIHRwX29wID0gaW50KG5wLmNvdW50X25vbnplcm8odHBfa1tzZWxdKSkKICAgIGZwX29wID0gaW50KG5wLmNvdW50X25vbnplcm8ofnRwX2tbc2VsXSkpCiAgICBmbl9vcCA9IG5waWcgLSB0cF9vcAogICAgcHJlYyA9IHRwX29wIC8gKHRwX29wICsgZnBfb3ApIGlmICh0cF9vcCArIGZwX29wKSBlbHNlIGZsb2F0KCJuYW4iKQogICAgcmVjID0gdHBfb3AgLyBucGlnCiAgICBmMSA9IDIgKiB0cF9vcCAvICgyICogdHBfb3AgKyBmcF9vcCArIGZuX29wKQoKICAgIGJlc3QgPSBkaWN0KGJlc3RfZjE9MC4wLCBiZXN0X2YxX2NvbmY9ZmxvYXQoIm5hbiIpLCBwcmVjaXNpb25fYXRfYmVzdF9mMT1mbG9hdCgibmFuIiksIHJlY2FsbF9hdF9iZXN0X2YxPTAuMCkKICAgIGlmIG5fcHJlZDoKICAgICAgICBjdHAgPSBucC5jdW1zdW0odHBfaykuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICAgICAgY2ZwID0gbnAuY3Vtc3VtKH50cF9rKS5hc3R5cGUobnAuZmxvYXQ2NCkKICAgICAgICBsYXN0X29mX3RpZSA9IG5wLnJfW3Nfa1sxOl0gIT0gc19rWzotMV0sIFRydWVdICAjIGEgY3V0IGlzIG9ubHkgcG9zc2libGUgYWZ0ZXIgYSBncm91cCBvZiBlcXVhbCBzY29yZXMKICAgICAgICBjdXQgPSBucC5ub256ZXJvKGxhc3Rfb2ZfdGllKVswXQogICAgICAgIGYxcyA9IDIgKiBjdHBbY3V0XSAvIChjdHBbY3V0XSArIGNmcFtjdXRdICsgbnBpZykKICAgICAgICBqID0gaW50KG5wLmFyZ21heChmMXMpKQogICAgICAgIGsgPSBjdXRbal0KICAgICAgICBiZXN0ID0gZGljdChiZXN0X2YxPV9mKGYxc1tqXSksIGJlc3RfZjFfY29uZj1fZihzX2tba10pLAogICAgICAgICAgICAgICAgICAgIHByZWNpc2lvbl9hdF9iZXN0X2YxPV9mKGN0cFtrXSAvIChjdHBba10gKyBjZnBba10pKSwgcmVjYWxsX2F0X2Jlc3RfZjE9X2YoY3RwW2tdIC8gbnBpZykpCiAgICAgICAgbWF4X3JlY2FsbCA9IF9mKGN0cFstMV0gLyBucGlnKQogICAgZWxzZToKICAgICAgICBtYXhfcmVjYWxsID0gMC4wCgogICAgIyByYXcgUFIgY3VydmUgKGFsbCBub24taWdub3JlZCBkZXRzLCBjb25mLXNvcnRlZCkKICAgIHJjdiwgcHJ2LCBjdiA9IHJjMFtrZWVwXSwgcHIwW2tlZXBdLCBzX2sKICAgIGlmIGxlbihyY3YpID4gX01BWF9QUl9QT0lOVFM6CiAgICAgICAgaWR4ID0gbnAudW5pcXVlKG5wLmxpbnNwYWNlKDAsIGxlbihyY3YpIC0gMSwgX01BWF9QUl9QT0lOVFMpLnJvdW5kKCkuYXN0eXBlKGludCkpCiAgICAgICAgcmN2LCBwcnYsIGN2ID0gcmN2W2lkeF0sIHBydltpZHhdLCBjdltpZHhdCgogICAgcmV0dXJuIHsKICAgICAgICAibl9pbWFnZXMiOiBuX2ltZywgIm5fZ3QiOiBucGlnLCAibl9wcmVkIjogbl9wcmVkLAogICAgICAgICJhcDUwIjogYXBfcGVyWzAuNV0gaWYgMC41IGluIGFwX3BlciBlbHNlIGZsb2F0KCJuYW4iKSwKICAgICAgICAiYXAiOiBmbG9hdChucC5tZWFuKGxpc3QoYXBfcGVyLnZhbHVlcygpKSkpLAogICAgICAgICJhcF9wZXJfaW91IjogYXBfcGVyLCAib3BfY29uZiI6IGZsb2F0KG9wX2NvbmYpLAogICAgICAgICJ0cCI6IHRwX29wLCAiZnAiOiBmcF9vcCwgImZuIjogZm5fb3AsCiAgICAgICAgInByZWNpc2lvbiI6IGZsb2F0KHByZWMpLCAicmVjYWxsIjogZmxvYXQocmVjKSwgImYxIjogZmxvYXQoZjEpLAogICAgICAgICoqYmVzdCwgIm1heF9yZWNhbGwiOiBtYXhfcmVjYWxsLAogICAgICAgICJwcl9jdXJ2ZSI6IHsicmVjYWxsIjogW2Zsb2F0KHgpIGZvciB4IGluIHJjdl0sICJwcmVjaXNpb24iOiBbZmxvYXQoeCkgZm9yIHggaW4gcHJ2XSwKICAgICAgICAgICAgICAgICAgICAgImNvbmYiOiBbZmxvYXQoeCkgZm9yIHggaW4gY3ZdfSwKICAgIH0KCgpkZWYgZXZhbHVhdGVfc2l6ZXMoZ3RzLCBwcmVkcywgKiwgYXJlYV9zY2FsZXM9Tm9uZSwgaW91X3RocnM9KDAuNSwpLCBtYXhfZGV0PTMwMCwgb3BfY29uZj0wLjI1LAogICAgICAgICAgICAgICAgICAgY2xhc3NfaWQ9Tm9uZSkgLT4gZGljdDoKICAgICIiInsiYWxsInwic21hbGwifCJtZWRpdW0ifCJsYXJnZSI6IGV2YWx1YXRlKC4uLil9IHVzaW5nIGNvbW1vbi5BUkVBX1JBTkdFUy4iIiIKICAgIHJldHVybiB7azogZXZhbHVhdGUoZ3RzLCBwcmVkcywgaW91X3RocnM9aW91X3RocnMsIGFyZWFfcm5nPXJuZywgYXJlYV9zY2FsZXM9YXJlYV9zY2FsZXMsCiAgICAgICAgICAgICAgICAgICAgICAgIG1heF9kZXQ9bWF4X2RldCwgb3BfY29uZj1vcF9jb25mLCBjbGFzc19pZD1jbGFzc19pZCkKICAgICAgICAgICAgZm9yIGssIHJuZyBpbiBjb21tb24uQVJFQV9SQU5HRVMuaXRlbXMoKX0KCgpkZWYgbWF0Y2hfZGV0YWlscyhndDogbnAubmRhcnJheSwgcHJlZDogbnAubmRhcnJheSwgKiwgaW91X3Rocj0wLjUsIG9wX2NvbmY9MC4yNSwgYmdfaW91PTAuMSwKICAgICAgICAgICAgICAgICAgY2xhc3NfYWdub3N0aWM9VHJ1ZSkgLT4gZGljdDoKICAgICIiIlBlci1ib3ggQ09DTyBncmVlZHkgbWF0Y2hpbmcgZm9yIG9uZSBpbWFnZSAocHJlZHMgd2l0aCBjb25mID49IG9wX2NvbmYsIG5vIGFyZWEgaWdub3JlKS4iIiIKICAgIGd0ID0gbnAuYXNhcnJheShndCwgZHR5cGU9bnAuZmxvYXQ2NCkucmVzaGFwZSgtMSwgNSkKICAgIHByZWQgPSBucC5hc2FycmF5KHByZWQsIGR0eXBlPW5wLmZsb2F0NjQpLnJlc2hhcGUoLTEsIDYpCiAgICBjYW5kID0gbnAubm9uemVybyhwcmVkWzosIDRdID49IG9wX2NvbmYpWzBdCiAgICBvcmRlciA9IGNhbmRbbnAuYXJnc29ydCgtcHJlZFtjYW5kLCA0XSwga2luZD0ibWVyZ2Vzb3J0IildCiAgICBwID0gcHJlZFtvcmRlcl0KICAgIGlvdXMgPSBib3hfaW91KHBbOiwgMDo0XSwgZ3RbOiwgMTo1XSkKICAgIGlmIG5vdCBjbGFzc19hZ25vc3RpYyBhbmQgaW91cy5zaXplOgogICAgICAgIGlvdXMgPSBucC53aGVyZShwWzosIDVdWzosIE5vbmVdID09IGd0WzosIDBdW05vbmUsIDpdLCBpb3VzLCAwLjApCiAgICBtID0gX2NvY29fbWF0Y2goaW91cywgbnAuemVyb3MobGVuKGd0KSwgZHR5cGU9bnAuaW50NjQpLCBpb3VfdGhyKQogICAgbmcgPSBsZW4oZ3QpCiAgICBndF9tYXRjaGVkID0gbnAuemVyb3MobmcsIGR0eXBlPWJvb2wpCiAgICBndF9jb25mID0gbnAuZnVsbChuZywgbnAubmFuKQogICAgc3RhdHVzID0gW10KICAgIGZvciBkIGluIHJhbmdlKGxlbihwKSk6CiAgICAgICAgaWYgbVtkXSA+PSAwOgogICAgICAgICAgICBndF9tYXRjaGVkW21bZF1dID0gVHJ1ZQogICAgICAgICAgICBndF9jb25mW21bZF1dID0gcFtkLCA0XQogICAgICAgICAgICBzdGF0dXMuYXBwZW5kKCJ0cCIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbXggPSBmbG9hdChpb3VzW2RdLm1heCgpKSBpZiBuZyBlbHNlIDAuMAogICAgICAgIGlmIG14ID49IG1pbihpb3VfdGhyLCAxIC0gMWUtMTApOgogICAgICAgICAgICBzdGF0dXMuYXBwZW5kKCJkdXBsaWNhdGUiKQogICAgICAgIGVsaWYgbXggPj0gYmdfaW91OgogICAgICAgICAgICBzdGF0dXMuYXBwZW5kKCJsb2NhbGl6YXRpb24iKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHN0YXR1cy5hcHBlbmQoImJhY2tncm91bmQiKQogICAgZ3RfYmVzdCA9IGlvdXMubWF4KGF4aXM9MCkgaWYgbGVuKHApIGFuZCBuZyBlbHNlIG5wLnplcm9zKG5nKQogICAgcmV0dXJuIHsiZ3RfbWF0Y2hlZCI6IGd0X21hdGNoZWQsICJndF9tYXRjaF9jb25mIjogZ3RfY29uZiwgImd0X2Jlc3RfaW91IjogZ3RfYmVzdC5hc3R5cGUobnAuZmxvYXQ2NCksCiAgICAgICAgICAgICJwcmVkX3N0YXR1cyI6IHN0YXR1cywgInByZWRfaW5kZXgiOiBvcmRlci5hc3R5cGUobnAuaW50NjQpfQo=", "phase5/predict.py": "IiIiVWx0cmFseXRpY3MgWU9MTyBpbmZlcmVuY2UgZm9yIFBoYXNlIDUgd2l0aCBhIEpTT04gcHJlZGljdGlvbiBjYWNoZS4KCkltYWdlcyBhcmUgYWx3YXlzIGRlY29kZWQgd2l0aCBgY29tbW9uLmltcmVhZGAgKHVuaWNvZGUtc2FmZSkgYW5kIGhhbmRlZCB0byBVbHRyYWx5dGljcyBhcyBCR1IgbnVtcHkKYXJyYXlzLCBzbyBVbHRyYWx5dGljcyBuZXZlciB0b3VjaGVzIHRoZSAoUGVyc2lhbikgZmlsZSBwYXRocyBpdHNlbGYuCk91dHB1dCBwZXIgaW1hZ2U6IGZsb2F0NjQgKE0sIDYpID0gW3gxLCB5MSwgeDIsIHkyLCBjb25mLCBjbHNdIGluIHRoZSBkZWNvZGVkIGltYWdlJ3MgcGl4ZWwgZnJhbWUuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKaW1wb3J0IHdhcm5pbmdzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIHBoYXNlNSBpbXBvcnQgY29tbW9uCmZyb20gcGhhc2U1LmNvbW1vbiBpbXBvcnQgRU1QVFlfUFJFRCwgSW1hZ2VSZWNvcmQKCkxPRyA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQpfQkFEX1RBU0tTID0geyJjbGFzc2lmeSIsICJwb3NlIiwgIm9iYiJ9CkNBQ0hFX1ZFUlNJT04gPSAyCgoKZGVmIF9sb2FkX21vZGVsKG1vZGVsX3BhdGg6IHN0cik6CiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCgogICAgbW9kZWwgPSBZT0xPKHN0cihtb2RlbF9wYXRoKSkKICAgIHRhc2sgPSBnZXRhdHRyKG1vZGVsLCAidGFzayIsIE5vbmUpCiAgICBpZiB0YXNrIGluIF9CQURfVEFTS1M6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm1vZGVsIHRhc2sgJ3t0YXNrfScgaXMgbm90IHN1cHBvcnRlZCAobmVlZCBkZXRlY3Qgb3Igc2VnbWVudCk6IHttb2RlbF9wYXRofSIpCiAgICByZXR1cm4gbW9kZWwKCgpkZWYgX21vZGVsX2ltZ3N6KG1vZGVsKToKICAgIGltZ3N6ID0gTm9uZQogICAgY2twdCA9IGdldGF0dHIobW9kZWwsICJja3B0IiwgTm9uZSkKICAgIGlmIGlzaW5zdGFuY2UoY2twdCwgZGljdCk6CiAgICAgICAgdGEgPSBja3B0LmdldCgidHJhaW5fYXJncyIpIG9yIHt9CiAgICAgICAgaWYgaXNpbnN0YW5jZSh0YSwgZGljdCk6CiAgICAgICAgICAgIGltZ3N6ID0gdGEuZ2V0KCJpbWdzeiIpCiAgICBpZiBub3QgaW1nc3o6CiAgICAgICAgaW1nc3ogPSAoZ2V0YXR0cihtb2RlbCwgIm92ZXJyaWRlcyIsIE5vbmUpIG9yIHt9KS5nZXQoImltZ3N6IikKICAgIGlmIGlzaW5zdGFuY2UoaW1nc3osIChsaXN0LCB0dXBsZSkpOgogICAgICAgIGltZ3N6ID0gbWF4KGludCh2KSBmb3IgdiBpbiBpbWdzeikgaWYgaW1nc3ogZWxzZSBOb25lCiAgICByZXR1cm4gaW50KGltZ3N6KSBpZiBpbWdzeiBlbHNlIE5vbmUKCgpkZWYgX2luZm9fZnJvbV9tb2RlbChtb2RlbCwgbW9kZWxfcGF0aDogc3RyKSAtPiBkaWN0OgogICAgbmFtZXMgPSBnZXRhdHRyKG1vZGVsLCAibmFtZXMiLCBOb25lKSBvciB7fQogICAgaWYgaXNpbnN0YW5jZShuYW1lcywgKGxpc3QsIHR1cGxlKSk6CiAgICAgICAgbmFtZXMgPSBkaWN0KGVudW1lcmF0ZShuYW1lcykpCiAgICBuYW1lcyA9IHtpbnQoayk6IHN0cih2KSBmb3IgaywgdiBpbiBuYW1lcy5pdGVtcygpfQogICAgcmV0dXJuIHsKICAgICAgICAibmFtZXMiOiBuYW1lcywKICAgICAgICAibmMiOiBsZW4obmFtZXMpLAogICAgICAgICJ0YXNrIjogc3RyKGdldGF0dHIobW9kZWwsICJ0YXNrIiwgImRldGVjdCIpKSwKICAgICAgICAiaW1nc3oiOiBfbW9kZWxfaW1nc3oobW9kZWwpLAogICAgICAgICJzaGExIjogY29tbW9uLmZpbGVfc2hhMShtb2RlbF9wYXRoKSwKICAgIH0KCgpkZWYgbW9kZWxfaW5mbyhtb2RlbF9wYXRoOiBzdHIpIC0+IGRpY3Q6CiAgICAiIiJ7Im5hbWVzIjoge2ludDogc3RyfSwgIm5jIiwgInRhc2siLCAiaW1nc3oiICh0cmFpbiBpbWdzeiBvciBOb25lKSwgInNoYTEiIChvZiB0aGUgbW9kZWwgLyB5YW1sIGZpbGUpfS4iIiIKICAgIG1vZGVsID0gX2xvYWRfbW9kZWwobW9kZWxfcGF0aCkKICAgIHJldHVybiBfaW5mb19mcm9tX21vZGVsKG1vZGVsLCBtb2RlbF9wYXRoKQoKCmRlZiBfaW1hZ2VzX3NoYTEocGF0aHM6IGxpc3Rbc3RyXSkgLT4gc3RyOgogICAgaCA9IGhhc2hsaWIuc2hhMSgpCiAgICBmb3IgcCBpbiBzb3J0ZWQocGF0aHMpOgogICAgICAgIHRyeToKICAgICAgICAgICAgc2l6ZSA9IG9zLnBhdGguZ2V0c2l6ZShwKQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICBzaXplID0gLTEKICAgICAgICBoLnVwZGF0ZShmIntwfVx0e3NpemV9XG4iLmVuY29kZSgidXRmLTgiKSkKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgoKZGVmIF9yZWFkX2NhY2hlKGNhY2hlX3BhdGg6IHN0ciwgbWV0YTogZGljdCwgaW1hZ2VzOiBsaXN0W3N0cl0pOgogICAgcCA9IFBhdGgoY2FjaGVfcGF0aCkKICAgIGlmIG5vdCBwLmlzX2ZpbGUoKToKICAgICAgICByZXR1cm4gTm9uZQogICAgdHJ5OgogICAgICAgIHdpdGggb3BlbihwLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBkYXRhID0ganNvbi5sb2FkKGYpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGU6CiAgICAgICAgTE9HLndhcm5pbmcoInByZWRpY3Rpb24gY2FjaGUgJXMgdW5yZWFkYWJsZSAoJXMpOyByZWNvbXB1dGluZyIsIHAsIGUpCiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGlmIG5vdCBpc2luc3RhbmNlKGRhdGEsIGRpY3QpIG9yIGRhdGEuZ2V0KCJtZXRhIikgIT0gbWV0YToKICAgICAgICBMT0cuaW5mbygicHJlZGljdGlvbiBjYWNoZSAlcyBtZXRhZGF0YSBtaXNtYXRjaDsgcmVjb21wdXRpbmciLCBwKQogICAgICAgIHJldHVybiBOb25lCiAgICBwcmVkcyA9IGRhdGEuZ2V0KCJwcmVkcyIpIG9yIHt9CiAgICBpZiBzZXQocHJlZHMpICE9IHNldChpbWFnZXMpOgogICAgICAgIHJldHVybiBOb25lCiAgICBvdXQgPSB7fQogICAgZm9yIGsgaW4gaW1hZ2VzOgogICAgICAgIGEgPSBucC5hc2FycmF5KHByZWRzW2tdLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgICAgIG91dFtrXSA9IGEucmVzaGFwZSgtMSwgNikgaWYgYS5zaXplIGVsc2UgRU1QVFlfUFJFRC5jb3B5KCkKICAgIHJldHVybiBvdXQKCgpkZWYgX3Jlc3VsdF90b19hcnJheShyKSAtPiBucC5uZGFycmF5OgogICAgYiA9IGdldGF0dHIociwgImJveGVzIiwgTm9uZSkKICAgIGlmIGIgaXMgTm9uZSBvciBsZW4oYikgPT0gMDoKICAgICAgICByZXR1cm4gRU1QVFlfUFJFRC5jb3B5KCkKICAgIHh5eHkgPSBiLnh5eHkuZGV0YWNoKCkuY3B1KCkubnVtcHkoKS5hc3R5cGUobnAuZmxvYXQ2NCkucmVzaGFwZSgtMSwgNCkKICAgIGNvbmYgPSBiLmNvbmYuZGV0YWNoKCkuY3B1KCkubnVtcHkoKS5hc3R5cGUobnAuZmxvYXQ2NCkucmVzaGFwZSgtMSwgMSkKICAgIGNscyA9IGIuY2xzLmRldGFjaCgpLmNwdSgpLm51bXB5KCkuYXN0eXBlKG5wLmZsb2F0NjQpLnJlc2hhcGUoLTEsIDEpCiAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUoW3h5eHksIGNvbmYsIGNsc10sIGF4aXM9MSkKCgpkZWYgcnVuX3ByZWRpY3Rpb25zKG1vZGVsX3BhdGg6IHN0ciwgcmVjb3JkczogbGlzdFtJbWFnZVJlY29yZF0sICosIGltZ3N6PTY0MCwgY29uZj0wLjAwMSwgaW91PTAuNywKICAgICAgICAgICAgICAgICAgICBhZ25vc3RpY19ubXM9VHJ1ZSwgbWF4X2RldD0zMDAsIGJhdGNoPTE2LCBkZXZpY2U9IiIsIGhhbGY9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgY2FjaGVfcGF0aDogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Rbc3RyLCBucC5uZGFycmF5XToKICAgICIiIlByZWRpY3QgZXZlcnkgcmVjb3JkIG9uY2U7IHJldHVybnMge3JlY29yZC5pbWFnZTogZmxvYXQ2NCAoTSwgNil9LiBVc2VzL3JlZnJlc2hlcyBgY2FjaGVfcGF0aGAuIiIiCiAgICBpbWFnZXMgPSBsaXN0KGRpY3QuZnJvbWtleXMoci5pbWFnZSBmb3IgciBpbiByZWNvcmRzKSkKICAgIGlzX3lhbWwgPSBzdHIobW9kZWxfcGF0aCkubG93ZXIoKS5lbmRzd2l0aCgoIi55YW1sIiwgIi55bWwiKSkKICAgIGlmIGNhY2hlX3BhdGggYW5kIHN0cihtb2RlbF9wYXRoKS5sb3dlcigpLmVuZHN3aXRoKCgiLnlhbWwiLCAiLnltbCIpKToKICAgICAgICB3YXJuaW5ncy53YXJuKGYibW9kZWwge21vZGVsX3BhdGh9IGlzIGEgLnlhbWwgKHJhbmRvbSB3ZWlnaHRzIHBlciBsb2FkKTsgcHJlZGljdGlvbiBjYWNoZSBkaXNhYmxlZCIpCiAgICAgICAgTE9HLndhcm5pbmcoInByZWRpY3Rpb24gY2FjaGUgZGlzYWJsZWQgZm9yIHlhbWwgbW9kZWwgJXMiLCBtb2RlbF9wYXRoKQogICAgICAgIGNhY2hlX3BhdGggPSBOb25lCiAgICBtZXRhID0gewogICAgICAgICJ2ZXJzaW9uIjogQ0FDSEVfVkVSU0lPTiwKICAgICAgICAibW9kZWxfc2hhMSI6IE5vbmUgaWYgaXNfeWFtbCBlbHNlIGNvbW1vbi5maWxlX3NoYTEobW9kZWxfcGF0aCksCiAgICAgICAgImltZ3N6IjogaW50KGltZ3N6KSwKICAgICAgICAiY29uZiI6IGZsb2F0KGNvbmYpLAogICAgICAgICJpb3UiOiBmbG9hdChpb3UpLAogICAgICAgICJhZ25vc3RpY19ubXMiOiBib29sKGFnbm9zdGljX25tcyksCiAgICAgICAgIm1heF9kZXQiOiBpbnQobWF4X2RldCksCiAgICAgICAgImhhbGYiOiBib29sKGhhbGYpLAogICAgICAgICJkZXZpY2UiOiBzdHIoZGV2aWNlKSwKICAgICAgICAiaW1hZ2VzX3NoYTEiOiBfaW1hZ2VzX3NoYTEoaW1hZ2VzKSwKICAgIH0KICAgIGlmIGNhY2hlX3BhdGg6CiAgICAgICAgY2FjaGVkID0gX3JlYWRfY2FjaGUoY2FjaGVfcGF0aCwgbWV0YSwgaW1hZ2VzKQogICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgTE9HLmluZm8oInVzaW5nIHByZWRpY3Rpb24gY2FjaGUgJXMgKCVkIGltYWdlcykiLCBjYWNoZV9wYXRoLCBsZW4oY2FjaGVkKSkKICAgICAgICAgICAgcmV0dXJuIGNhY2hlZAoKICAgIG1vZGVsID0gX2xvYWRfbW9kZWwobW9kZWxfcGF0aCkKICAgIG91dDogZGljdFtzdHIsIG5wLm5kYXJyYXldID0ge30KICAgIGJhdGNoID0gbWF4KDEsIGludChiYXRjaCkpCiAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oaW1hZ2VzKSwgYmF0Y2gpOgogICAgICAgIGNodW5rID0gaW1hZ2VzW2kgOiBpICsgYmF0Y2hdCiAgICAgICAgYXJycywga2V5cyA9IFtdLCBbXQogICAgICAgIGZvciBwIGluIGNodW5rOgogICAgICAgICAgICBpbWcgPSBjb21tb24uaW1yZWFkKHApCiAgICAgICAgICAgIGlmIGltZyBpcyBOb25lOgogICAgICAgICAgICAgICAgd2FybmluZ3Mud2FybihmInVucmVhZGFibGUgaW1hZ2UsIGVtcHR5IHByZWRpY3Rpb24gdXNlZDoge3B9IikKICAgICAgICAgICAgICAgIExPRy53YXJuaW5nKCJ1bnJlYWRhYmxlIGltYWdlOiAlcyIsIHApCiAgICAgICAgICAgICAgICBvdXRbcF0gPSBFTVBUWV9QUkVELmNvcHkoKQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgYXJycy5hcHBlbmQoaW1nKQogICAgICAgICAgICBrZXlzLmFwcGVuZChwKQogICAgICAgIGlmIG5vdCBhcnJzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJlc3VsdHMgPSBtb2RlbC5wcmVkaWN0KHNvdXJjZT1hcnJzLCBpbWdzej1pbWdzeiwgY29uZj1jb25mLCBpb3U9aW91LCBhZ25vc3RpY19ubXM9YWdub3N0aWNfbm1zLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9kZXQ9bWF4X2RldCwgZGV2aWNlPShOb25lIGlmIGRldmljZSA9PSAiIiBlbHNlIGRldmljZSksIGhhbGY9aGFsZiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2ZXJib3NlPUZhbHNlLCBzdHJlYW09RmFsc2UpCiAgICAgICAgaWYgbGVuKHJlc3VsdHMpICE9IGxlbihrZXlzKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVWx0cmFseXRpY3MgcmV0dXJuZWQge2xlbihyZXN1bHRzKX0gcmVzdWx0cyBmb3Ige2xlbihrZXlzKX0gaW1hZ2VzIikKICAgICAgICBmb3IgaywgciBpbiB6aXAoa2V5cywgcmVzdWx0cyk6CiAgICAgICAgICAgIG91dFtrXSA9IF9yZXN1bHRfdG9fYXJyYXkocikKICAgIG91dCA9IHtrOiBvdXRba10gZm9yIGsgaW4gaW1hZ2VzfQogICAgaWYgY2FjaGVfcGF0aDoKICAgICAgICBjb21tb24uZHVtcF9qc29uKGNhY2hlX3BhdGgsIHsibWV0YSI6IG1ldGEsICJwcmVkcyI6IHtrOiB2LnRvbGlzdCgpIGZvciBrLCB2IGluIG91dC5pdGVtcygpfX0pCiAgICByZXR1cm4gb3V0CgoKZGVmIGNsYXNzaWZ5X2RldGVjdGlvbnMoY2xzX21vZGVsX3BhdGg6IHN0ciwgcmVjb3JkczogbGlzdFtJbWFnZVJlY29yZF0sIGRldHM6IGRpY3Rbc3RyLCBucC5uZGFycmF5XSwgKiwKICAgICAgICAgICAgICAgICAgICAgICAgbWluX2NvbmY6IGZsb2F0ID0gMC4wMSwgaW1nc3o6IGludCA9IDE2MCwgYmF0Y2g6IGludCA9IDEyOCwgZGV2aWNlPSIiLAogICAgICAgICAgICAgICAgICAgICAgICBjYWNoZV9wYXRoOiBzdHIgfCBOb25lID0gTm9uZSkgLT4gZGljdFtzdHIsIG5wLm5kYXJyYXldOgogICAgIiIiU3RhZ2UgMjogY2xhc3NpZnkgZXZlcnkgZGV0ZWN0ZWQgYm94IChjb25mID49IG1pbl9jb25mKSB3aXRoIGEgWU9MTyBjbGFzc2lmeSBtb2RlbC4KCiAgICBSZXR1cm5zIHtpbWFnZTogKE0sIDYpfSB3aXRoIGNscyA9IHRvcC0xIGNsYXNzIG9mIHRoZSBjcm9wIGFuZCBjb25mID0gZGV0X2NvbmYgKiBwKHRvcC0xKSwgdGhlIHVzdWFsCiAgICB0d28tc3RhZ2Ugc2NvcmUsIHNvIDQtY2xhc3MgQVAgY2FuIGJlIGNvbXB1dGVkLiBDcm9wcyB1c2UgcHJlcGFyZV9kYXRhLmNyb3BfYm94IChzYW1lIHJ1bGUgYXMgdHJhaW5pbmcpLgogICAgIiIiCiAgICBmcm9tIHBoYXNlNS5wcmVwYXJlX2RhdGEgaW1wb3J0IGNyb3BfYm94CgogICAgaW1hZ2VzID0gbGlzdChkaWN0LmZyb21rZXlzKHIuaW1hZ2UgZm9yIHIgaW4gcmVjb3JkcykpCiAgICBtZXRhID0geyJ2ZXJzaW9uIjogQ0FDSEVfVkVSU0lPTiwgImNsc19zaGExIjogY29tbW9uLmZpbGVfc2hhMShjbHNfbW9kZWxfcGF0aCksICJtaW5fY29uZiI6IGZsb2F0KG1pbl9jb25mKSwKICAgICAgICAgICAgImltZ3N6IjogaW50KGltZ3N6KSwgImltYWdlc19zaGExIjogX2ltYWdlc19zaGExKGltYWdlcyksCiAgICAgICAgICAgICJkZXRzX3NoYTEiOiBoYXNobGliLnNoYTEoYiIiLmpvaW4obnAuYXNjb250aWd1b3VzYXJyYXkoZGV0cy5nZXQoaSwgY29tbW9uLkVNUFRZX1BSRUQpKS50b2J5dGVzKCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGkgaW4gaW1hZ2VzKSkuaGV4ZGlnZXN0KCl9CiAgICBpZiBjYWNoZV9wYXRoOgogICAgICAgIGNhY2hlZCA9IF9yZWFkX2NhY2hlKGNhY2hlX3BhdGgsIG1ldGEsIGltYWdlcykKICAgICAgICBpZiBjYWNoZWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybiBjYWNoZWQKICAgIGZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KCiAgICBtb2RlbCA9IFlPTE8oY2xzX21vZGVsX3BhdGgpCiAgICBpZiBtb2RlbC50YXNrICE9ICJjbGFzc2lmeSI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntjbHNfbW9kZWxfcGF0aH0gaXMgYSAne21vZGVsLnRhc2t9JyBtb2RlbCwgZXhwZWN0ZWQgYSBjbGFzc2lmeSBtb2RlbCIpCiAgICBvdXQ6IGRpY3Rbc3RyLCBucC5uZGFycmF5XSA9IHt9CiAgICBmb3IgaW1fcGF0aCBpbiBpbWFnZXM6CiAgICAgICAgZCA9IG5wLmFzYXJyYXkoZGV0cy5nZXQoaW1fcGF0aCwgY29tbW9uLkVNUFRZX1BSRUQpLCBkdHlwZT1ucC5mbG9hdDY0KS5yZXNoYXBlKC0xLCA2KQogICAgICAgIGQgPSBkW2RbOiwgNF0gPj0gbWluX2NvbmZdCiAgICAgICAgaW1nID0gY29tbW9uLmltcmVhZChpbV9wYXRoKSBpZiBsZW4oZCkgZWxzZSBOb25lCiAgICAgICAgaWYgaW1nIGlzIE5vbmUgb3Igbm90IGxlbihkKToKICAgICAgICAgICAgb3V0W2ltX3BhdGhdID0gY29tbW9uLkVNUFRZX1BSRUQuY29weSgpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgY3JvcHMsIGtlZXAgPSBbXSwgW10KICAgICAgICBmb3IgaSwgYiBpbiBlbnVtZXJhdGUoZCk6CiAgICAgICAgICAgIGNyID0gY3JvcF9ib3goaW1nLCBiWzo0XSwgc2l6ZT1pbWdzeikKICAgICAgICAgICAgaWYgY3IgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBjcm9wcy5hcHBlbmQoY3IpCiAgICAgICAgICAgICAgICBrZWVwLmFwcGVuZChpKQogICAgICAgIHJvd3MgPSBbXQogICAgICAgIGZvciBzIGluIHJhbmdlKDAsIGxlbihjcm9wcyksIGJhdGNoKToKICAgICAgICAgICAgcmVzID0gbW9kZWwucHJlZGljdChjcm9wc1tzOnMgKyBiYXRjaF0sIGltZ3N6PWltZ3N6LCB2ZXJib3NlPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRldmljZT1Ob25lIGlmIGRldmljZSA9PSAiIiBlbHNlIGRldmljZSkKICAgICAgICAgICAgZm9yIGssIHIgaW4gZW51bWVyYXRlKHJlcyk6CiAgICAgICAgICAgICAgICBwID0gci5wcm9icwogICAgICAgICAgICAgICAgYiA9IGRba2VlcFtzICsga11dCiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZChbYlswXSwgYlsxXSwgYlsyXSwgYlszXSwgYls0XSAqIGZsb2F0KHAudG9wMWNvbmYpLCBpbnQocC50b3AxKV0pCiAgICAgICAgb3V0W2ltX3BhdGhdID0gbnAuYXNhcnJheShyb3dzLCBkdHlwZT1ucC5mbG9hdDY0KS5yZXNoYXBlKC0xLCA2KSBpZiByb3dzIGVsc2UgY29tbW9uLkVNUFRZX1BSRUQuY29weSgpCiAgICBpZiBjYWNoZV9wYXRoOgogICAgICAgIGNvbW1vbi5kdW1wX2pzb24oY2FjaGVfcGF0aCwgeyJtZXRhIjogbWV0YSwgInByZWRzIjoge2s6IHYudG9saXN0KCkgZm9yIGssIHYgaW4gb3V0Lml0ZW1zKCl9fSkKICAgIHJldHVybiBvdXQK", "phase5/prepare_data.py": "...", "phase5/train_cls.py": "IiIiU3RhZ2UtMiB0cmFpbmluZzogWU9MTyBjbGFzc2lmeSBtb2RlbCBvbiB0b21hdG8gY3JvcHMgKHJhdyAvIGhhbGZfcmlwZSAvIHJpcGUgLyByb3R0ZW4pLgoKVHJhaW5zIHN0cmFpZ2h0IGludG8gPHByb2plY3Q+L3J1biBvbiBHb29nbGUgRHJpdmUgKHNtYWxsIG1vZGVsOiBhIGNoZWNrcG9pbnQgZXZlcnkgZXBvY2ggaXMgY2hlYXApLCByZXN1bWVzCmZyb20gbGFzdC5wdCBhZnRlciBhIGRpc2Nvbm5lY3QsIHdyaXRlcyBET05FIHdoZW4gZmluaXNoZWQgKHRoZW4gYSByZXJ1biBkb2VzIG5vdGhpbmcpIGFuZCB0ZXN0Lmpzb24gd2l0aAp0aGUgYWNjdXJhY3kgb24gdGhlIFRvbWF0byBEZWZlY3RzIHRlc3QgY3JvcHMuCgogIHB5dGhvbiAtbSBwaGFzZTUudHJhaW5fY2xzIC0tZGF0YSAvY29udGVudC93b3JrL2Nsc190b21hdG8gLS1wcm9qZWN0IC9jb250ZW50L2RyaXZlLy4uLi9ydW5zL2NscyBcCiAgICAgIC0tbW9kZWwgeW9sbzI2cy1jbHMucHQgLS1pbWdzeiAxNjAgLS1lcG9jaHMgNjAKIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KCgpkZWYgbWFpbihhcmd2PU5vbmUpIC0+IGludDoKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXByb2plY3QiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1vZGVsIiwgZGVmYXVsdD0ieW9sbzI2cy1jbHMucHQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWltZ3N6IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTYwKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTYwKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXBhdGllbmNlIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZnJhY3Rpb24iLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTEuMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PU5vbmUpCiAgICBhID0gYXAucGFyc2VfYXJncyhhcmd2KQogICAgcnVuID0gUGF0aChhLnByb2plY3QpIC8gInJ1biIKICAgIGlmIChydW4gLyAiRE9ORSIpLmV4aXN0cygpOgogICAgICAgIHByaW50KCJbY2xzXSB0cmFpbmluZyBhbHJlYWR5IGZpbmlzaGVkOiIsIHJ1biAvICJ3ZWlnaHRzIiAvICJiZXN0LnB0IiwgZmx1c2g9VHJ1ZSkKICAgIGVsc2U6CiAgICAgICAgbGFzdCA9IHJ1biAvICJ3ZWlnaHRzIiAvICJsYXN0LnB0IgogICAgICAgIGlmIGxhc3QuZXhpc3RzKCk6CiAgICAgICAgICAgIHByaW50KCJbY2xzXSByZXN1bWluZyBmcm9tIiwgbGFzdCwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgWU9MTyhzdHIobGFzdCkpLnRyYWluKHJlc3VtZT1UcnVlKQogICAgICAgICAgICBleGNlcHQgQXNzZXJ0aW9uRXJyb3I6CiAgICAgICAgICAgICAgICBwcmludCgiW2Nsc10gbm90aGluZyB0byByZXN1bWU7IHRyZWF0aW5nIGFzIGZpbmlzaGVkIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmludChmIltjbHNdIG5ldyBydW46IHthLm1vZGVsfSwgaW1nc3oge2EuaW1nc3p9LCB7YS5lcG9jaHN9IGVwb2NocyIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIFlPTE8oYS5tb2RlbCkudHJhaW4oZGF0YT1hLmRhdGEsIGVwb2Nocz1hLmVwb2NocywgaW1nc3o9YS5pbWdzeiwgYmF0Y2g9YS5iYXRjaCwgcGF0aWVuY2U9YS5wYXRpZW5jZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwcm9qZWN0PXN0cihQYXRoKGEucHJvamVjdCkpLCBuYW1lPSJydW4iLCBleGlzdF9vaz1UcnVlLCB3b3JrZXJzPWEud29ya2VycywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmcmFjdGlvbj1hLmZyYWN0aW9uLCBkZXZpY2U9YS5kZXZpY2UsIHBsb3RzPVRydWUsIHNlZWQ9MCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmbGlwbHI9MC41LCBmbGlwdWQ9MC41LCBoc3ZfaD0wLjAxLCBoc3Zfcz0wLjQsIGhzdl92PTAuMywgZXJhc2luZz0wLjIpCiAgICAgICAgKHJ1biAvICJET05FIikud3JpdGVfdGV4dCgib2siKQogICAgYmVzdCA9IHJ1biAvICJ3ZWlnaHRzIiAvICJiZXN0LnB0IgogICAgbSA9IFlPTE8oc3RyKGJlc3QpKS52YWwoZGF0YT1hLmRhdGEsIHNwbGl0PSJ0ZXN0IiwgaW1nc3o9YS5pbWdzeiwgYmF0Y2g9YS5iYXRjaCwgcGxvdHM9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByb2plY3Q9c3RyKHJ1biksIG5hbWU9InRlc3RfZXZhbCIsIGV4aXN0X29rPVRydWUsIGRldmljZT1hLmRldmljZSwgdmVyYm9zZT1GYWxzZSkKICAgIHJlcyA9IHsidG9wMSI6IGZsb2F0KG0udG9wMSksICJ0b3A1IjogZmxvYXQobS50b3A1KX0KICAgIChydW4gLyAidGVzdC5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcyksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChmIltjbHNdIFRFU1QgYWNjdXJhY3kgKFRvbWF0byBEZWZlY3RzIHRlc3QgY3JvcHMpOiB0b3AxIHtyZXNbJ3RvcDEnXTouM2Z9IiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo="}
for name, blob in SCRIPTS.items():
    os.makedirs(os.path.dirname(f"{WORK}/{name}"), exist_ok=True)
    with open(f"{WORK}/{name}", "wb") as f:
        f.write(base64.b64decode(blob))
sys.path.insert(0, WORK)
import panel
importlib.reload(panel)
from IPython.display import display, HTML, Image
print("کدها آماده:", ", ".join(SCRIPTS))

ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")


def sysinfo():
    out = []
    try:
        q = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        u, mu, mt, t = [x.strip() for x in q.strip().split(",")]
        out.append(f"GPU {u}% busy · {float(mu) / 1024:.1f}/{float(mt) / 1024:.0f} GB · {t}°C")
    except Exception:
        pass
    try:
        import psutil
        vm = psutil.virtual_memory()
        out.append(f"CPU {psutil.cpu_percent():.0f}% · RAM {vm.used / 1e9:.1f}/{vm.total / 1e9:.1f} GB")
    except Exception:
        pass
    return " | ".join(out)


def train_live(cmd, status_file, log_file):
    """Run train_tomato.py and redraw the live panel every 2 s. Returns (exit code, last log lines, status)."""
    if os.path.exists(status_file):
        os.remove(status_file)
    tail = collections.deque(maxlen=14)
    with open(log_file, "a", encoding="utf-8") as lf:
        lf.write("\n$ " + " ".join(cmd) + "\n")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})

    def reader():                              # reads the output as it comes: \r progress bars included
        buf, t_bar = "", 0.0
        with open(log_file, "a", encoding="utf-8") as lf:
            while True:
                chunk = p.stdout.read1(65536)
                if not chunk:
                    break
                buf += chunk.decode("utf-8", "replace")
                parts = re.split(r"(\r|\n)", buf)
                buf = parts.pop()
                for seg, sep in zip(parts[0::2], parts[1::2]):
                    s = ANSI.sub("", seg).strip()
                    if not s:
                        continue
                    is_bar = "━" in s or "─" in s
                    if is_bar and tail and ("━" in tail[-1] or "─" in tail[-1]):
                        tail[-1] = s           # the same progress bar, redrawn
                    else:
                        tail.append(s)
                    if not is_bar or sep == "\n" or time.time() - t_bar > 30:
                        lf.write(s + "\n")
                        lf.flush()
                        t_bar = time.time() if is_bar else t_bar

    th = threading.Thread(target=reader, daemon=True)
    th.start()

    def status():
        try:
            return json.load(open(status_file, encoding="utf-8"))
        except Exception:
            return {"phase": "setup", "run": RUN_NAME, "model": MODEL_SIZE, "imgsz": RUN_IMGSZ, "epochs": RUN_EPOCHS,
                    "session_start": t0}

    t0 = time.time()
    h = display(HTML(panel.live_html(status(), list(tail))), display_id=True) or PanelFile(f"{WORK}/live_panel.html")
    n_hist, chart_html, info, t_info = -1, "", "", 0.0
    try:
        while True:
            rc = p.poll()
            st = status()
            hist = st.get("history") or []
            if len(hist) != n_hist:
                n_hist, chart_html = len(hist), panel.chart(hist, st.get("epochs"))
            if time.time() - t_info > 5:
                info, t_info = sysinfo(), time.time()
            if rc is not None:
                th.join(10)
                if rc != 0 and st.get("phase") not in ("stopped", "done"):
                    st["phase"] = "error"
            h.update(HTML(panel.live_html(st, list(tail), info, chart_html)))
            if rc is not None:
                return rc, list(tail), st
            time.sleep(2)
    except KeyboardInterrupt:
        print("در حال توقف ... آخرین epoch کامل روی Drive ذخیره می‌شود (حداکثر یک دقیقه)", flush=True)
        p.send_signal(signal.SIGINT)
        for _ in range(3):
            try:
                p.wait(timeout=90)
                break
            except (subprocess.TimeoutExpired, KeyboardInterrupt):
                p.kill()
        th.join(10)
        st = status()
        st["phase"] = "stopped"
        h.update(HTML(panel.live_html(st, list(tail), info, chart_html)))
        print("متوقف شد. برای ادامه: Run all (یا فقط بلوک‌ها را از اول اجرا کن).")
        raise
    except BaseException:                      # a problem in this loop must never leave the training running alone
        p.kill()
        raise


class PanelFile:
    """Outside a notebook display() returns nothing: the panel goes to an HTML file instead."""
    def __init__(self, path):
        self.path = path

    def update(self, obj):
        with open(self.path, "w", encoding="utf-8") as f:
            f.write('<meta charset="utf-8">' + obj.data)
os.makedirs(DRIVE_DIR, exist_ok=True)
RUNS = f"{DRIVE_DIR}/runs_test" if QUICK_TEST else f"{DRIVE_DIR}/runs"
PROJECT = f"{RUNS}/{MODEL}"
os.makedirs(PROJECT, exist_ok=True)
print("نتیجه‌ها روی Drive:", PROJECT)

## 2) داده: دانلود (یک بار)، holdout ثابت، ساخت داده‌ی آموزش، بررسی نشتی

In [ ]:
#@title 2) داده: دانلود و ساخت فقط بار اول، روی Google Drive
import yaml
def run_quiet(cmd, ok_msg):
    r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    if r.returncode != 0:
        print(chr(10).join((r.stdout + r.stderr).strip().splitlines()[-30:]))
        raise SystemExit(f"خطا در مرحله: {ok_msg} (خط‌های بالا علت را نوشته‌اند)")
    print("✅", ok_msg)
    return r.stdout
os.chdir(WORK)
from phase5 import prepare_data as prep
UNI = f"{WORK}/unified"                          # same path in every session (the holdout manifest relies on it)
DATA_ZIP = f"{DRIVE_DIR}/tomato_data_v2.zip"     # downloaded + converted ONCE, kept on Drive
if all(os.path.exists(f"{UNI}/{u}/.ready") for u in prep.UNIFIED):
    print("داده‌ها روی این جلسه آماده‌اند")
elif os.path.exists(DATA_ZIP):
    print("داده‌ها از Google Drive باز می‌شوند (دانلود دوباره لازم نیست) ...", flush=True)
    shutil.rmtree(UNI, ignore_errors=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(WORK)
else:
    print("بار اول: دانلود ۶ دیتاست از Roboflow (چند دقیقه) ...", flush=True)
    prep.download_all(f"{WORK}/raw", ROBOFLOW_API_KEY.strip())
    print("یکسان‌سازی کلاس‌ها ...", flush=True)
    prep.build_unified(f"{WORK}/raw", UNI)
    shutil.make_archive(DATA_ZIP[:-4] + "_part", "zip", WORK, "unified")
    os.replace(DATA_ZIP[:-4] + "_part.zip", DATA_ZIP)
    shutil.rmtree(f"{WORK}/raw", ignore_errors=True)
    print("✅ داده‌ها روی Google Drive ذخیره شدند:", DATA_ZIP)
for u, fa in (("u_defects", "(الف) Tomato Defects"), ("u_sxsc7", "(ب) tomato-sxsc7"), ("u_conveyor", "(ج) نوار نقاله"),
              ("u_extra", "داده‌ی کمکی آموزش")):
    st = json.load(open(f"{UNI}/{u}/.ready", encoding="utf-8"))
    print(f"   {fa}: {st['images']} عکس | کادرها: " + "، ".join(f"{k} {v}" for k, v in st["boxes"].items() if v))

CFG = f"{DRIVE_DIR}/phase5_config.yaml"
CLS_PROJECT = f"{RUNS}/cls_{CLS_MODEL[:-3]}"
cfg = {"model": f"{PROJECT}/final/best.pt", "classifier": f"{CLS_PROJECT}/run/weights/best.pt", "cls_imgsz": CLS_IMGSZ,
       "imgsz": RUN_IMGSZ, "batch": 16, "device": "",
       "out_dir": f"{DRIVE_DIR}/reports/phase5", "splits_dir": f"{DRIVE_DIR}/splits",
       "cache_dir": f"{DRIVE_DIR}/reports/phase5/pred_cache",
       "datasets": {"defects": {"root": f"{UNI}/u_defects", "eval_split": "test", "tomato_classes": "all"},
                    "sxsc7": {"root": f"{UNI}/u_sxsc7", "tomato_classes": "all",
                              "holdout": {"use_existing_split": "test", "min_images": 30, "frac": 0.2}},
                    "conveyor": {"root": f"{UNI}/u_conveyor", "tomato_classes": "all", "holdout": {"n_images": 30}}}}
yaml.safe_dump(cfg, open(CFG, "w"), sort_keys=False)
# frozen holdout: made once before the first training; later runs only verify it (never re-split)
first = not os.path.exists(f"{DRIVE_DIR}/splits/manifest.json")
if first:
    print("جدا کردن holdout (فقط بار اول، حدود ۲ دقیقه) ...", flush=True)
run_quiet([sys.executable, "-m", "phase5.make_holdouts", "--config", CFG],
          "holdout ساخته شد" if first else "holdout ثابت بررسی شد (تغییری نکرده)")
_m = json.load(open(f"{DRIVE_DIR}/splits/manifest.json", encoding="utf-8"))
for _k, _fa in (("conveyor", "نوار نقاله"), ("sxsc7", "tomato-sxsc7")):
    _v = _m[_k]
    print(f"   {_fa}: holdout {len(_v['holdout'])} عکس (هرگز آموزش نمی‌بیند) | آموزش {len(_v['train'])} عکس"
          f" | کنار گذاشته به‌خاطر شباهت با holdout {len(_v.get('excluded', []))}")

# images of the Tomato Defects train/valid (and extra data) that are near-copies of its test images: removed
from pathlib import Path as _P
bad = prep.clean_train_images(_P(UNI), cache=_P(f"{DRIVE_DIR}/splits/near_duplicates_removed.json"))
DATA = f"{WORK}/final_{MODEL}"                  # stage 1 (detector): one class "tomato"
counts = prep.build_detector_set(UNI, f"{DRIVE_DIR}/splits", DATA, bad)
print(f"مرحله‌ی ۱ (تشخیص گوجه): آموزش {counts['train']} | اعتبارسنجی {counts['val']} | آزمون {counts['test']} عکس"
      f" | حذف‌شده به‌خاطر کپی بودن از test: {counts['removed_near_duplicates']}")
CLS_DATA = f"{WORK}/cls_{MODEL}"                # stage 2 (ripeness): crops, built once and kept on Drive
CLS_ZIP = f"{DRIVE_DIR}/tomato_cls_v2.zip"
if not os.path.exists(f"{CLS_DATA}/.ready") and os.path.exists(CLS_ZIP):
    with zipfile.ZipFile(CLS_ZIP) as z:
        z.extractall(WORK)
if not os.path.exists(f"{CLS_DATA}/.ready"):
    print("ساخت تصاویر برش‌خورده‌ی گوجه برای مرحله‌ی ۲ (فقط بار اول) ...", flush=True)
    prep.build_classifier_set(UNI, CLS_DATA, bad=bad)
    shutil.make_archive(CLS_ZIP[:-4] + "_part", "zip", WORK, os.path.basename(CLS_DATA))
    os.replace(CLS_ZIP[:-4] + "_part.zip", CLS_ZIP)
cc = json.load(open(f"{CLS_DATA}/.ready"))
print("مرحله‌ی ۲ (رسیدگی):", " | ".join(f"{ {'train': 'آموزش', 'val': 'اعتبارسنجی', 'test': 'آزمون'}[k]}: " +
      "، ".join(f"{n} {v}" for n, v in d.items()) for k, d in cc.items()))
LEAK_OK = f"{DRIVE_DIR}/splits/leakage_ok_{counts['train']}_{counts['val']}.txt"
if os.path.exists(LEAK_OK):
    print("✅ بررسی نشتی قبلاً انجام شده: هیچ عکس holdout در train/val نیست")
else:
    print("بررسی نشتی (فقط بار اول، حدود ۲ دقیقه) ...", flush=True)
    run_quiet([sys.executable, "-m", "phase5.check_leakage", "--config", CFG, "--train", f"{DATA}/data.yaml"],
              "بررسی نشتی: هیچ عکس holdout در train/val نیست")
    open(LEAK_OK, "w").write("ok")

## 3) وضعیت آموزش
چند epoch انجام شده، بهترین دقت، زمان باقی‌مانده و این‌که آموزش از کجا ادامه پیدا می‌کند.

In [ ]:
#@title 3) وضعیت آموزش
display(HTML(panel.status_html(PROJECT, RUN_NAME, {"model": MODEL_SIZE, "imgsz": RUN_IMGSZ, "epochs": RUN_EPOCHS,
                                                   "patience": PATIENCE})))

## 4) آموزش — پنل زنده
پنل هر ۲ ثانیه به‌روز می‌شود. توقف دستی: ■ (آخرین epoch کامل روی Drive ذخیره می‌شود و دفعه‌ی بعد از همان‌جا ادامه می‌دهد).

In [ ]:
#@title 4) آموزش با پنل زنده
status_file, log_file = f"{WORK}/live_status.json", f"{WORK}/train_log.txt"
try:                                           # a training left over from an earlier stopped cell: stop it cleanly
    if subprocess.run(["pkill", "-INT", "-f", "train_tomato.py"], capture_output=True).returncode == 0:
        print("یک آموزش قبلی هنوز در حال اجرا بود؛ متوقفش می‌کنم ...", flush=True)
        for _ in range(60):
            time.sleep(2)
            if subprocess.run(["pgrep", "-f", "train_tomato.py"], capture_output=True).returncode != 0:
                break
        subprocess.run(["pkill", "-KILL", "-f", "train_tomato.py"], capture_output=True)
except FileNotFoundError:
    pass
cmd =[sys.executable, f"{WORK}/train_tomato.py", "--data", DATA, "--work", f"{WORK}/runs", "--drive", PROJECT,
       "--model", MODEL_SIZE, "--imgsz", str(RUN_IMGSZ), "--epochs", str(RUN_EPOCHS), "--patience", str(PATIENCE),
       "--status", status_file, "--save-every", "1" if QUICK_TEST else str(SAVE_EVERY)]
if QUICK_TEST:
    cmd += ["--fraction", "0.03", "--cache", "False"]
cmd += ["--workers", str(max(2, min(16, os.cpu_count() or 2)))]           # every CPU core loads data
if not torch.cuda.is_available():                                           # no GPU: train on the CPU
    cmd += ["--device", "cpu"]
batch = int(BATCH)
for attempt in range(3):
    log_start = os.path.getsize(log_file) if os.path.exists(log_file) else 0
    rc, tail, st = train_live(cmd + (["--batch", str(batch)] if batch else []), status_file, log_file)
    if rc == 0:
        break
    with open(log_file, encoding="utf-8", errors="replace") as f:
        f.seek(log_start)
        log = f.read()
    if "out of memory" in log.lower() and attempt < 2:      # resume with a smaller batch
        batch = max(2, int(st.get("batch") or 8) // 2)
        print(f"حافظه‌ی GPU کم آمد؛ ادامه با batch {batch} ...", flush=True)
        continue
    print("\n".join(log.strip().splitlines()[-25:]))
    raise RuntimeError(f"آموزش با خطا تمام شد (کد {rc}) - خط‌های بالا علت را نوشته‌اند")
if QUICK_TEST:
    print("\nآزمایش سریع بی‌خطا تمام شد. در تنظیمات QUICK_TEST را خاموش کن و دوباره Run all بزن تا آموزش اصلی شروع شود.")

## 4.ب) آموزش مرحله‌ی ۲ — تشخیص رسیدگی (raw / half_ripe / ripe / rotten) روی برش هر گوجه
سریع است (حدود ۲۰ تا ۴۰ دقیقه). اگر تمام شده باشد، دوباره انجام نمی‌شود؛ اگر قطع شده باشد، ادامه می‌دهد.

In [ ]:
#@title 4.ب) آموزش مرحله‌ی ۲ (رسیدگی)
cmd = [sys.executable, "-m", "phase5.train_cls", "--data", CLS_DATA, "--project", CLS_PROJECT, "--model", CLS_MODEL,
       "--imgsz", str(CLS_IMGSZ), "--epochs", "2" if QUICK_TEST else str(CLS_EPOCHS),
       "--workers", str(max(2, min(8, os.cpu_count() or 2)))]
if QUICK_TEST:
    cmd += ["--fraction", "0.05"]
os.chdir(WORK)
run_live(cmd)
for f in (f"{CLS_PROJECT}/run/results.png", f"{CLS_PROJECT}/run/test_eval/confusion_matrix_normalized.png"):
    if os.path.exists(f):
        display(Image(f, width=850))

## 5) نتیجه‌ی ۴ کلاسه روی test (عکس‌هایی که مدل هرگز ندیده)

In [ ]:
#@title 5) نتیجه‌ی ۴ کلاسه
FINAL = f"{PROJECT}/final"
if not os.path.exists(f"{FINAL}/summary.json"):
    print("هنوز آموزش تمام نشده")
else:
    s = json.load(open(f"{FINAL}/summary.json"))
    run = s["chosen"]
    ev = json.load(open(f"{PROJECT}/{run}/eval.json"))
    display(HTML(panel.eval_html(ev)))
    if len(s["runs"]) > 1:
        print("همه‌ی آموزش‌ها (mAP50-95):", json.dumps(s["runs"]), "| انتخاب‌شده با val:", run)
    for pat, w in [("results.png", 950), ("test_eval/confusion_matrix_normalized.png", 720), ("test_eval/*PR_curve.png", 720)]:
        for f in sorted(glob.glob(f"{PROJECT}/{run}/{pat}"))[:1]:
            display(Image(f, width=w))
    out = f"{WORK}/test_preview_{run}"
    if not glob.glob(out + "/*_annotated.jpg"):
        by_src = collections.defaultdict(list)       # test photos with defects, a few from every source
        for p in sorted(glob.glob(f"{DATA}/images/test/*.jpg")):
            lbl = p.replace("/images/", "/labels/")[:-4] + ".txt"
            if os.path.exists(lbl) and os.path.getsize(lbl) > 0:
                by_src[os.path.basename(p).split("__")[0]].append(p)
        picks = []
        while len(picks) < 8 and any(by_src.values()):
            for k in list(by_src):
                if by_src[k] and len(picks) < 8:
                    lst = by_src[k]
                    picks.append(lst.pop(len(lst) // 2))
        prev = f"{WORK}/preview_src"
        shutil.rmtree(prev, ignore_errors=True)
        os.makedirs(prev)
        for p in picks:
            shutil.copy2(p, prev)
        run_live([sys.executable, f"{WORK}/predict.py", "--weights", f"{FINAL}/best.pt", "--source", prev,
                  "--out", out, "--labels", f"{DATA}/labels/test"])
    for p in sorted(glob.glob(out + "/*_annotated.jpg")):
        display(Image(p, width=620))
    print("بهترین مدل:", f"{FINAL}/best.pt", "| آستانه‌ها:", f"{FINAL}/thresholds.json")

## 6) ارزیابی class-agnostic «گوجه در برابر پس‌زمینه» (خواسته‌ی اصلی)
سه مجموعه: test دیتاست Defects، holdout از sxsc7، holdout نوار نقاله (≥۳۰ عکس که هرگز آموزش ندیده‌اند). پیش‌بینی‌ها cache می‌شوند و دوبار حساب نمی‌شوند.

In [ ]:
#@title 6) ارزیابی گوجه در برابر پس‌زمینه (خواسته‌ی اصلی)
import math
if not os.path.exists(f"{PROJECT}/final/best.pt"):
    print("اول آموزش باید تمام شود")
else:
    os.chdir(WORK)
    OUT = f"{DRIVE_DIR}/reports/phase5"
    print("ارزیابی روی سه مجموعه ... (پیش‌بینی‌های قبلی از cache خوانده می‌شوند)", flush=True)
    run_quiet([sys.executable, "-m", "phase5.evaluate_class_agnostic", "--config", CFG, "--crosscheck"],
              "ارزیابی class-agnostic تمام شد")
    R = json.load(open(f"{OUT}/class_agnostic_results.json", encoding="utf-8"))
    def f3(x):
        return "—" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.3f}"
    FA_SET = {"defects_test": "(الف) test دیتاست Tomato Defects", "sxsc7_holdout": "(ب) holdout از tomato-sxsc7",
              "conveyor_holdout": "(ج) نوار نقاله (holdout، هرگز آموزش ندیده)"}
    css = "<div dir='rtl' style='font-family:Tahoma,sans-serif;font-size:14px'>"
    g = R["gate"]; rc = g.get("recall")
    ok = rc is not None and not math.isnan(rc) and rc >= g["threshold"]
    html = css + (f"<h3 style='color:#1a7f37'>✅ قبول: recall کلاس-مستقل روی نوار نقاله {f3(rc)} ≥ {g['threshold']}</h3>" if ok else
                  f"<h3 style='color:#c62828'>❌ زیر هدف: recall کلاس-مستقل روی نوار نقاله {f3(rc)} &lt; {g['threshold']} — "
                  f"یعنی مدل بخشی از گوجه‌های روی نوار را پیدا نمی‌کند (علت‌ها پایین‌تر)</h3>")
    op = R["sets"]["defects_test"]["summary"].get("op_conf")
    html += f"<h3>جدول پذیرش — گوجه در برابر پس‌زمینه (همه‌ی ۴ کلاس = یک کلاس «گوجه»)</h3><table border=1 cellpadding=5 style='border-collapse:collapse'>"
    html += f"<tr><th>مجموعه</th><th>تعداد عکس</th><th>تعداد گوجه</th><th>mAP50</th><th>mAP50-95</th><th>Precision (conf {op})</th><th>Recall (conf {op})</th><th>F1</th><th>بیشترین recall ممکن</th><th>mAP50 تأیید Ultralytics</th></tr>"
    for k, lab in FA_SET.items():
        s = R["sets"][k]; m = s["summary"]; cc = s.get("crosscheck") or {}
        html += (f"<tr><td>{lab}</td><td>{s['n_images']}</td><td>{s['n_gt_tomatoes']}</td><td><b>{f3(m['ap50'])}</b></td>"
                 f"<td>{f3(m['ap50_95'])}</td><td><b>{f3(m['precision'])}</b></td><td><b>{f3(m['recall'])}</b></td><td>{f3(m['f1'])}</td>"
                 f"<td>{f3(m['max_recall'])}</td><td>{f3(cc.get('map50'))}</td></tr>")
    html += "</table>"
    FA_SZ = {"small": "کوچک (&lt;۳۲×۳۲ پیکسل)", "medium": "متوسط (۳۲ تا ۹۶)", "large": "بزرگ (&gt;۹۶)"}
    for key, ttl in (("sizes_original_px", "اندازه بر حسب پیکسل عکس اصلی"), ("sizes_input_px", f"اندازه بر حسب پیکسل ورودی شبکه ({RUN_IMGSZ})")):
        html += f"<h3>عملکرد بر اساس اندازه‌ی گوجه — {ttl}</h3><table border=1 cellpadding=5 style='border-collapse:collapse'><tr><th>مجموعه</th>"
        html += "".join(f"<th>{FA_SZ[b]}: تعداد / AP50 / recall</th>" for b in FA_SZ) + "</tr>"
        for k, lab in FA_SET.items():
            sz = R["sets"][k][key]
            html += f"<tr><td>{lab}</td>" + "".join(f"<td>{sz[b].get('n_gt') or 0} / {f3(sz[b]['ap50'])} / {f3(sz[b]['recall'])}</td>" for b in FA_SZ) + "</tr>"
        html += "</table>"
    fc = R["sets"]["defects_test"].get("four_class")
    if fc:
        html += "<h3>متریک‌های ۴ کلاسه (دو مرحله: تشخیص + رسیدگی) روی test (الف)</h3><table border=1 cellpadding=5 style='border-collapse:collapse'><tr><th>کلاس</th><th>AP50</th><th>AP50-95</th><th>Precision</th><th>Recall</th></tr>"
        for n, v in fc["per_class"].items():
            html += f"<tr><td>{n}</td><td>{f3(v['ap50'])}</td><td>{f3(v['ap50_95'])}</td><td>{f3(v['precision'])}</td><td>{f3(v['recall'])}</td></tr>"
        mm = fc["mean"]
        html += f"<tr><td><b>میانگین</b></td><td><b>{f3(mm['ap50'])}</b></td><td>{f3(mm['ap50_95'])}</td><td>{f3(mm['precision'])}</td><td>{f3(mm['recall'])}</td></tr></table>"
    fl = R["sets"]["conveyor_holdout"].get("failures") or {}
    FA_FN = {"small_object": "گوجه‌ی کوچک / دور", "motion_blur": "محوشدگی حرکتی", "occlusion": "پوشیده شدن / بریده شدن در لبه‌ی عکس",
             "belt_background": "هم‌رنگی با نوار (کنتراست کم)", "other": "علت دیگر"}
    FA_FP = {"belt_background_fp": "تشخیص اشتباه روی نوار / پس‌زمینه", "localization_fp": "کادر نادقیق", "duplicate_fp": "کادر تکراری"}
    html += (f"<h3>تحلیل خطا روی نوار نقاله</h3><p>گوجه‌های پیدا نشده: <b>{fl.get('n_fn', 0)}</b> | تشخیص‌های اشتباه: <b>{fl.get('n_fp', 0)}</b>"
             "<br>(یک گوجه می‌تواند چند علت داشته باشد)</p><table border=1 cellpadding=5 style='border-collapse:collapse'><tr><th>نوع خطا</th><th>تعداد</th><th>سهم از گوجه‌های پیدا نشده</th></tr>")
    for t, n in FA_FN.items():
        html += f"<tr><td>{n}</td><td>{(fl.get('fn_tags') or {}).get(t, 0)}</td><td>{100 * (fl.get('fn_tag_share') or {}).get(t, 0):.0f}%</td></tr>"
    for t, n in FA_FP.items():
        html += f"<tr><td>{n}</td><td>{(fl.get('fp_tags') or {}).get(t, 0)}</td><td>—</td></tr>"
    html += "</table><p>رنگ‌ها در عکس‌ها: سبز = گوجه‌ی پیدا شده، قرمز = گوجه‌ی پیدا نشده (با علت)، نارنجی = تشخیص اشتباه، آبی = پیش‌بینی درست</p></div>"
    display(HTML(html))
    def absp(p):
        return p if os.path.isabs(p) else os.path.join(WORK, p)
    for t, n in {**FA_FN, **FA_FP}.items():
        ex = (fl.get("examples") or {}).get(t) or []
        if not ex:
            continue
        display(HTML(f"<h4 dir='rtl' style='font-family:Tahoma'>نمونه‌ها: {n} ({len(ex)} عکس)</h4>"))
        sheet = (fl.get("contact_sheets") or {}).get(t)
        if sheet and os.path.exists(absp(sheet)):
            display(Image(absp(sheet), width=900))
        for p in ex[:3]:
            if os.path.exists(absp(p)):
                display(Image(absp(p), width=700))
    print("همه‌ی نمونه‌ها و گزارش کامل روی Drive:", OUT)

## 7) تست روی عکس خودت
`IMAGE` خالی = دکمه‌ی آپلود.

In [ ]:
#@title 7) تست روی عکس خودت
IMAGE = ""   #@param {type:"string"}
CONF  = 0   #@param {type:"number"}
weights = f"{PROJECT}/final/best.pt"
if not os.path.exists(weights):
    print("اول آموزش باید تمام شود:", weights)
else:
    if not IMAGE and IN_COLAB:
        from google.colab import files
        up = files.upload(); IMAGE = f"{WORK}/upload"; os.makedirs(IMAGE, exist_ok=True)
        for n, b in up.items():
            open(f"{IMAGE}/{n}", "wb").write(b)
    if IMAGE:
        out = f"{WORK}/my_test_{MODEL}"
        run_live([sys.executable, f"{WORK}/predict.py", "--weights", weights, "--source", IMAGE, "--out", out]
                 + (["--conf", str(CONF)] if CONF else []))
        for p in sorted(glob.glob(out + "/*_annotated.jpg")):
            display(Image(p, width=1100))